# SIF Regression: Drought × Irrigation Interaction — CONUS

**Research question:** Does irrigation buffering reduce SIF anomaly losses under drought, and where does this effect operate across CONUS cropland?

This notebook implements an empirical statistical model inspired by He & Rosa (2026, *Nature Food*), who used panel regression with irrigation–climate interaction terms to isolate irrigation effects on crop yield. We adapt the same interaction-term framework but with remote sensing observables:

- **Response variable:** SIF z-score (`sif_z`) — solar-induced fluorescence anomaly relative to the 2015–2024 pixel-month climatology. Negative values indicate below-normal photosynthetic activity.
- **Drought predictor:** SPEI-90d (`spei90d`) — 90-day standardized precipitation-evapotranspiration index. Negative = drought.
- **Irrigation predictor:** HumanET / ΔET (`delta_et`) — OpenET minus NLDAS Noah ET (mm/month). Positive values indicate net anthropogenic water additions, primarily irrigation.
- **Interaction term:** SPEI × ΔET — tests whether the SIF response to drought differs between irrigated and non-irrigated pixels.

**The model:**

$$\text{SIF}_z = \beta_1 \cdot \text{SPEI} + \beta_2 \cdot \Delta\text{ET} + \beta_3 \cdot (\text{SPEI} \times \Delta\text{ET}) + \alpha_m + \varepsilon$$

Month fixed effects ($\alpha_m$) absorb the seasonal SIF cycle so coefficients reflect anomalies.

**Key hypothesis (β₃):**
- β₃ < 0 → irrigation is *more* effective at maintaining SIF when drought is severe (buffering effect)
- β₃ > 0 → irrigation benefit diminishes under drought conditions

**Prerequisites:** Run `01_human_et_conus.ipynb` first to generate `data/processed/conus/regression/df_combined_gs.parquet`.


In [1]:
# Standard scientific Python stack
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats

# statsmodels provides OLS regression with heteroskedasticity-robust standard errors
import statsmodels.formula.api as smf

# Resolve project root from environment variable or relative path
_root_env    = os.environ.get('SIF_ROOT')
project_root = Path(_root_env) if _root_env else Path('../../..').resolve()

proc = project_root / 'data' / 'processed' / 'conus'
figs = project_root / 'figures' / 'conus'
figs.mkdir(parents=True, exist_ok=True)

print('Project root:', project_root)
print('Data dir:    ', proc)
print('Figures dir: ', figs)


Project root: /home/pielab-sandbox-jcoldiron/SIF-Analysis
Data dir:     /home/pielab-sandbox-jcoldiron/SIF-Analysis/data/processed/conus
Figures dir:  /home/pielab-sandbox-jcoldiron/SIF-Analysis/figures/conus


## 1. Configuration

Key thresholds carried over from the main data notebook. These must match the values used when building the dataset in `01_human_et_conus.ipynb`.


In [2]:
# ── CONUS processed data grid (matches 01_human_et_conus.ipynb) ──────────────────────────────
# 0.125° resolution, 325 columns × 189 rows (agricultural CONUS domain)
# Extent: lon -124.6875 to -84.1875 W, lat 49.3125 to 25.8125 N
CONUS_LON = np.arange(-124.6875, -84.0625, 0.125)   # 325 longitudes
CONUS_LAT = np.arange(  49.3125,  25.6875, -0.125)  # 189 latitudes (N→S)
n_lat, n_lon = len(CONUS_LAT), len(CONUS_LON)

# ── Analysis parameters ──────────────────────────────────────────────────────────────────
YEARS          = list(range(2015, 2025))    # 10-year study period
GROWING_SEASON = [4, 5, 6, 7, 8, 9]        # April–September

# Irrigation threshold: HumanET > 20 mm/month = high-confidence irrigation
IRR_THRESHOLD_MM = 20.0

# Drought threshold for pixel-level spatial analysis
DROUGHT_SPEI_THRESH = -0.5   # SPEI ≤ -0.5 = at least mild drought (D0 equivalent)

# Minimum observations per pixel for pixel-level regression
MIN_OBS_PIXEL = 5

# Significance threshold for spatial map filtering
ALPHA_SPATIAL = 0.10   # p < 0.10 for pixel-level slopes (liberal, for mapping)

print('CONUS grid:', n_lon, 'x', n_lat, 'at 0.125 deg')
print('Study period:', YEARS[0], '-', YEARS[-1])
print('Growing season months:', GROWING_SEASON)
print('Irrigation threshold:', IRR_THRESHOLD_MM, 'mm/mo')
print('Drought threshold (SPEI):', DROUGHT_SPEI_THRESH)


CONUS grid: 325 x 189 at 0.125 deg
Study period: 2015 - 2024
Growing season months: [4, 5, 6, 7, 8, 9]
Irrigation threshold: 20.0 mm/mo
Drought threshold (SPEI): -0.5


## 2. Load Pre-processed Data

The combined pixel-month dataset was assembled in `01_human_et_conus.ipynb` using **all CONUS cropland pixels** (~25,000 cells) across all growing-season months (April–September, 2015–2024). Each row is one pixel × one month observation with co-located measurements of SIF anomaly, HumanET, and drought indices.

We also reload the static cropland mask (189 × 325 boolean array) needed for the spatial maps in Section 7.

**Columns in `df_combined_gs.parquet`:**

| Column | Description |
|--------|-------------|
| `year`, `month` | Temporal index |
| `lat`, `lon` | Pixel center coordinates |
| `sif_z` | SIF z-score (anomaly relative to pixel-month climatology) |
| `delta_et` | HumanET = OpenET − NLDAS Noah ET (mm/month) |
| `is_irrigated` | 1 if delta_et > 20 mm/mo, 0 otherwise |
| `spei90d` | 90-day SPEI drought index |
| `dm_cat` | USDM-equivalent drought category (-1=none, 0=D0 ... 4=D4) |


In [3]:
# ── Load the pixel-month panel dataset ───────────────────────────────────────────────────
parquet_path = proc / 'regression' / 'df_combined_gs.parquet'
mask_path    = proc / 'regression' / 'crop_mask_static.npy'

if not parquet_path.exists():
    raise FileNotFoundError(
        'Run 01_human_et_conus.ipynb first to generate: ' + str(parquet_path)
    )

df = pd.read_parquet(parquet_path)

# Re-derive date column (parquet may or may not preserve datetime dtype)
if 'date' not in df.columns:
    df['date'] = pd.to_datetime(df['yyyymm'], format='%Y%m')

# Ensure dm_cat_int exists for grouping
if 'dm_cat_int' not in df.columns and 'dm_cat' in df.columns:
    df['dm_cat_int'] = df['dm_cat'].round().astype('Int64')

print('Loaded:', parquet_path.name)
print('Shape:', df.shape)
print('Columns:', list(df.columns))
print()
print('Date range:', df['date'].min().date(), 'to', df['date'].max().date())
print('Unique pixels (lat/lon pairs):', df.groupby(['lat','lon']).ngroups)
print('Unique year-months:', df['yyyymm'].nunique())
print()
print('Missing values per column:')
print(df.isnull().sum().to_string())

# ── Load static cropland mask ──────────────────────────────────────────────────────────────────
if mask_path.exists():
    crop_mask_static = np.load(mask_path)
    print()
    print('Cropland mask shape:', crop_mask_static.shape)
    print('Cropland cells:', crop_mask_static.sum(), '/', crop_mask_static.size)
else:
    print('WARNING: crop_mask_static.npy not found. Spatial maps will be unmasked.')
    crop_mask_static = None


Loaded: df_combined_gs.parquet
Shape: (1509180, 12)
Columns: ['year', 'month', 'yyyymm', 'lat', 'lon', 'sif_z', 'delta_et', 'is_irrigated', 'dm_cat', 'spei90d', 'dm_cat_int', 'date']

Date range: 2015-04-01 to 2024-09-01
Unique pixels (lat/lon pairs): 25153
Unique year-months: 60

Missing values per column:
year                  0
month                 0
yyyymm                0
lat                   0
lon                   0
sif_z            708985
delta_et        1066225
is_irrigated    1066225
dm_cat                0
spei90d               0
dm_cat_int            0
date                  0

Cropland mask shape: (189, 325)
Cropland cells: 25153 / 61425


## 3. Exploratory Data Summary

Before fitting the model, we examine the joint distribution of the three main variables (SIF z-score, SPEI, and ΔET) and confirm that we have adequate drought coverage and irrigated pixel representation. The key concern is whether drought and irrigation co-occur sufficiently to estimate the interaction term.


In [4]:
# ── Summary statistics ─────────────────────────────────────────────────────────────────────────────
print('=== Key Variable Summary ===')
print(df[['sif_z', 'spei90d', 'delta_et', 'is_irrigated']].describe().round(3).to_string())
print()

# ── Drought × irrigation co-occurrence ────────────────────────────────────────────────────────
# We need enough drought + irrigated observations to estimate the interaction term.
df_clean = df.dropna(subset=['sif_z', 'spei90d', 'delta_et'])
print('Complete cases (no NaN in sif_z, spei90d, delta_et):', len(df_clean))
print()

# Cross-tabulate drought category vs irrigated flag
dm_labels = {-1: 'No drought', 0: 'D0', 1: 'D1', 2: 'D2', 3: 'D3', 4: 'D4'}
if 'dm_cat_int' in df_clean.columns:
    print('Observations by drought category and irrigation status:')
    ct = pd.crosstab(
        df_clean['dm_cat_int'].map(lambda x: dm_labels.get(int(x), str(x)) if pd.notna(x) else 'NaN'),
        df_clean['is_irrigated'].map({0: 'Rainfed', 1: 'Irrigated', np.nan: 'Unknown'}),
        margins=True
    )
    print(ct.to_string())
    print()

# ── Year × month coverage ──────────────────────────────────────────────────────────────────────────
print('Records per year (growing season):')
for yr in YEARS:
    n = (df_clean['year'] == yr).sum()
    print('  ' + str(yr) + ': ' + f'{n:>8,}')

# ── Quick distribution plot ──────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df_clean['sif_z'].dropna(), bins=60, color='#2a7b0f', edgecolor='none', alpha=0.8)
axes[0].axvline(0, color='k', linewidth=1, linestyle='--')
axes[0].set_xlabel('SIF z-score')
axes[0].set_ylabel('Count')
axes[0].set_title('SIF Z-score Distribution')
axes[0].grid(alpha=0.3)

axes[1].hist(df_clean['spei90d'].dropna(), bins=60, color='#2196F3', edgecolor='none', alpha=0.8)
axes[1].axvline(0, color='k', linewidth=1, linestyle='--')
axes[1].set_xlabel('SPEI-90d')
axes[1].set_title('SPEI-90d Distribution')
axes[1].grid(alpha=0.3)

axes[2].hist(df_clean['delta_et'].clip(-50, 200).dropna(), bins=60, color='#FF8C00', edgecolor='none', alpha=0.8)
axes[2].axvline(IRR_THRESHOLD_MM, color='red', linewidth=1.5, linestyle='--',
                label='Irrigation threshold')
axes[2].set_xlabel('HumanET / \u0394ET (mm/mo)')
axes[2].set_title('\u0394ET Distribution (clipped at 200)')
axes[2].legend(fontsize=9)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(figs / 'reg_variable_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reg_variable_distributions.png')


=== Key Variable Summary ===


            sif_z      spei90d    delta_et  is_irrigated
count  800195.000  1509180.000  442955.000    442955.000
mean       -0.062        0.023      22.258         0.472
std         0.976        0.514      25.466         0.499
min        -3.984       -2.090    -133.371         0.000
25%        -0.772        0.000       6.241         0.000
50%        -0.119        0.000      18.621         0.000
75%         0.617        0.000      33.355         1.000
max         4.227        2.090     233.769         1.000

Complete cases (no NaN in sif_z, spei90d, delta_et): 434245

Observations by drought category and irrigation status:


is_irrigated  Irrigated  Rainfed     All
dm_cat_int                              
D0                36479    30729   67208
D1                23354    15399   38753
D2                 8736     4797   13533
D3                  859      354    1213
No drought       135415   178123  313538
All              204843   229402  434245

Records per year (growing season):
  2015:   43,841
  2016:   44,394
  2017:   36,985
  2018:   44,788
  2019:   45,003
  2020:   44,941
  2021:   44,912
  2022:   44,769
  2023:   44,763
  2024:   39,849


Saved: reg_variable_distributions.png


## 4. Statistical Model

### Model specification

We estimate a pooled ordinary least squares (OLS) regression with month fixed effects:

$$\text{SIF}_z = \beta_1 \cdot \text{SPEI} + \beta_2 \cdot \Delta\text{ET} + \beta_3 \cdot (\text{SPEI} \times \Delta\text{ET}) + \alpha_m + \varepsilon$$

**Why pooled OLS with month FE (rather than pixel FE)?**
- Month fixed effects ($\alpha_m$) absorb the seasonal cycle of SIF, which dominates spatial variation. Since SIF z-scores are already pixel-month standardized anomalies, month FE primarily controls for residual aggregate anomalies common to all pixels in a given month (e.g., cloud cover artifacts, sensor gaps).
- Pixel fixed effects would absorb all time-invariant spatial structure, but with only ~60 observations per pixel (6 months × 10 years), pixel FE estimates would be noisy and are sensitive to missing data gaps from OCO-2 orbital sampling. A panel estimator is preferred as a robustness check.
- HC3 heteroskedasticity-robust standard errors account for variance differences across irrigated vs. rainfed pixels without assuming equal residual variance.

### Interpretation of the interaction term β₃

The marginal effect of ΔET on SIF_z is not constant — it varies with drought severity:

$$\frac{\partial \text{SIF}_z}{\partial \Delta\text{ET}} = \beta_2 + \beta_3 \cdot \text{SPEI}$$

- Under severe drought (SPEI = −2): ME = β₂ + β₃·(−2)
- Under normal conditions (SPEI = 0): ME = β₂
- Under wet conditions (SPEI = +1): ME = β₂ + β₃·(+1)

If **β₃ < 0**, then irrigation has a *larger* positive effect on SIF when drought is severe (SPEI very negative), which is the irrigation buffering hypothesis. If β₃ > 0, irrigation is *less* effective under drought.

### Data preparation

We drop rows missing any of the three key variables, create the interaction term, and use the month as a categorical fixed effect.


In [5]:
# ── Build the regression-ready dataset ────────────────────────────────────────────────────────────────
# Drop rows missing any of the three model variables
df_reg = df.dropna(subset=['sif_z', 'spei90d', 'delta_et']).copy()

# Also drop any remaining non-finite values (inf, -inf can occur in derived fields)
df_reg = df_reg[
    np.isfinite(df_reg['sif_z']) &
    np.isfinite(df_reg['spei90d']) &
    np.isfinite(df_reg['delta_et'])
].copy()

# Create the interaction term: SPEI × ΔET
# When SPEI < 0 (drought) and ΔET > 0 (irrigation), this product is negative.
# The coefficient β₃ on this term captures the drought × irrigation interaction.
df_reg['spei_x_det'] = df_reg['spei90d'] * df_reg['delta_et']

# Month as string for fixed-effect formula (statsmodels C() notation)
df_reg['month_str'] = df_reg['month'].astype(str)

print('Regression dataset: ' + f'{len(df_reg):,}' + ' pixel-month observations')
print('Dropped: ' + str(len(df) - len(df_reg)) + ' rows with missing values')
print()
print('SPEI-90d:  mean=' + f'{df_reg["spei90d"].mean():.3f}' +
      '  std=' + f'{df_reg["spei90d"].std():.3f}' +
      '  range=[' + f'{df_reg["spei90d"].min():.2f}' + ', ' + f'{df_reg["spei90d"].max():.2f}' + ']')
print('\u0394ET:       mean=' + f'{df_reg["delta_et"].mean():.1f}' +
      '  std=' + f'{df_reg["delta_et"].std():.1f}' +
      '  range=[' + f'{df_reg["delta_et"].min():.1f}' + ', ' + f'{df_reg["delta_et"].max():.1f}' + '] mm/mo')
print('SIF z:     mean=' + f'{df_reg["sif_z"].mean():.3f}' +
      '  std=' + f'{df_reg["sif_z"].std():.3f}')
print()

# Report drought coverage
n_drought = (df_reg['spei90d'] <= DROUGHT_SPEI_THRESH).sum()
n_irr     = (df_reg['delta_et'] > IRR_THRESHOLD_MM).sum()
n_both    = ((df_reg['spei90d'] <= DROUGHT_SPEI_THRESH) & (df_reg['delta_et'] > IRR_THRESHOLD_MM)).sum()
print('Drought obs (SPEI \u2264 ' + str(DROUGHT_SPEI_THRESH) + '): ' + f'{n_drought:,}' +
      ' (' + f'{100*n_drought/len(df_reg):.1f}' + '%)')
print('Irrigated obs (\u0394ET > ' + str(IRR_THRESHOLD_MM) + ' mm/mo): ' + f'{n_irr:,}' +
      ' (' + f'{100*n_irr/len(df_reg):.1f}' + '%)')
print('Drought + irrigated: ' + f'{n_both:,}' +
      ' (' + f'{100*n_both/len(df_reg):.1f}' + '%) \u2190 these identify \u03b2\u2083')


Regression dataset: 434,245 pixel-month observations
Dropped: 1074935 rows with missing values

SPEI-90d:  mean=0.044  std=0.856  range=[-2.09, 2.09]
ΔET:       mean=22.2  std=25.4  range=[-133.4, 233.8] mm/mo
SIF z:     mean=-0.072  std=0.976

Drought obs (SPEI ≤ -0.5): 120,766 (27.8%)
Irrigated obs (ΔET > 20.0 mm/mo): 204,843 (47.2%)
Drought + irrigated: 69,453 (16.0%) ← these identify β₃


## 5. Regression Results

We fit the pooled OLS model using `statsmodels`. The formula uses `C(month)` to include month fixed effects as dummy variables (with one month dropped as the reference category). Standard errors are HC3 robust (heteroskedasticity-consistent, MacKinnon-White estimator), appropriate for cross-sectional data with potentially unequal variances across pixel types.


In [6]:
# ── Fit pooled OLS with month fixed effects ──────────────────────────────────────────────────────
# Formula: SIF_z ~ SPEI + deltaET + SPEI*deltaET + month dummies
# smf.ols uses Patsy formula syntax; C(month) creates treatment-coded dummies.
res = smf.ols(
    'sif_z ~ spei90d + delta_et + spei_x_det + C(month)',
    data=df_reg
).fit(cov_type='HC3')   # HC3 robust standard errors

# ── Extract key coefficients ──────────────────────────────────────────────────────────────────────────
b1, b2, b3 = res.params['spei90d'], res.params['delta_et'], res.params['spei_x_det']
p1, p2, p3 = res.pvalues['spei90d'], res.pvalues['delta_et'], res.pvalues['spei_x_det']

key_vars   = ['spei90d', 'delta_et', 'spei_x_det']
row_labels = ['SPEI-90d (drought)', '\u0394ET / HumanET', 'SPEI \u00d7 \u0394ET (interaction)']

coef_df = pd.DataFrame({
    'Coef.':    [res.params[v] for v in key_vars],
    'Std.Err.': [res.bse[v]    for v in key_vars],
    't':        [res.tvalues[v] for v in key_vars],
    'P>|t|':    [res.pvalues[v] for v in key_vars],
    'CI 2.5%':  [res.conf_int().loc[v, 0] for v in key_vars],
    'CI 97.5%': [res.conf_int().loc[v, 1] for v in key_vars],
}, index=row_labels)

# ── Print results ───────────────────────────────────────────────────────────────────────────────────
print('Pooled OLS + month fixed effects (HC3 robust SE)')
print('N = ' + f'{int(res.nobs):,}' +
      '   R\u00b2 = ' + f'{res.rsquared:.3f}' +
      '   Adj. R\u00b2 = ' + f'{res.rsquared_adj:.3f}')
print()
print(coef_df.to_string(float_format=lambda x: f'{x:.5f}'))
print()

# ── Interpret β₃ ────────────────────────────────────────────────────────────────────────────────
sig3 = '***' if p3 < 0.001 else ('**' if p3 < 0.01 else ('*' if p3 < 0.05 else 'n.s.'))
direction = ('NEGATIVE \u2014 irrigation buffers SIF losses under drought'
             if b3 < 0 else
             'POSITIVE \u2014 irrigation benefit diminishes under drought')
print('\u03b2\u2083 (interaction) = ' + f'{b3:.5f}' +
      '  (' + sig3 + ', p = ' + f'{p3:.4g}' + ')')
print('Direction: ' + direction)
print()

# ── Marginal effect of ΔET at different SPEI levels ──────────────────────────────────────────────
# This shows how the irrigation-SIF relationship changes across the drought gradient.
print('Conditional marginal effect of \u0394ET on SIF_z (b2 + b3 \u00d7 SPEI):')
print('  (Positive = more irrigation associated with higher SIF z-score)')
print()
for sv, lbl in [(-2.5, 'Extreme drought D4'), (-2.0, 'Severe drought D3'),
                (-1.5, 'Moderate drought D2'), (-0.5, 'Mild drought D0/D1'),
                ( 0.0, 'No drought'),          ( 1.0, 'Wet conditions')]:
    me  = b2 + b3 * sv
    sig = '*' if abs(me) > 1.96 * np.sqrt(res.bse['delta_et']**2 + sv**2 * res.bse['spei_x_det']**2) else ''
    print('  SPEI=' + f'{sv:+.1f}' + ' (' + lbl + '): ME = ' + f'{me:+.5f}' + ' SIF-z per mm/mo ' + sig)


Pooled OLS + month fixed effects (HC3 robust SE)
N = 434,245   R² = 0.206   Adj. R² = 0.206

                           Coef.  Std.Err.        t   P>|t|  CI 2.5%  CI 97.5%
SPEI-90d (drought)       0.17264   0.00209 82.73700 0.00000  0.16855   0.17673
ΔET / HumanET            0.00357   0.00006 62.94109 0.00000  0.00346   0.00368
SPEI × ΔET (interaction) 0.00067   0.00006 10.50611 0.00000  0.00055   0.00080

β₃ (interaction) = 0.00067  (***, p = 8.097e-26)
Direction: POSITIVE — irrigation benefit diminishes under drought

Conditional marginal effect of ΔET on SIF_z (b2 + b3 × SPEI):
  (Positive = more irrigation associated with higher SIF z-score)

  SPEI=-2.5 (Extreme drought D4): ME = +0.00189 SIF-z per mm/mo *
  SPEI=-2.0 (Severe drought D3): ME = +0.00223 SIF-z per mm/mo *
  SPEI=-1.5 (Moderate drought D2): ME = +0.00257 SIF-z per mm/mo *
  SPEI=-0.5 (Mild drought D0/D1): ME = +0.00324 SIF-z per mm/mo *
  SPEI=+0.0 (No drought): ME = +0.00357 SIF-z per mm/mo *
  SPEI=+1.0 (Wet condit

## 6. Coefficient Visualization

Two complementary figures summarize the regression:

1. **Coefficient bar chart** — shows the estimated main effects (β₁, β₂) and interaction (β₃) with 95% confidence intervals. Significance stars follow the standard convention (*** p<0.001, ** p<0.01, * p<0.05).

2. **Marginal effect curve** — plots how the effect of ΔET on SIF z-score (∂SIF_z/∂ΔET = β₂ + β₃·SPEI) varies continuously across the SPEI gradient. The shaded band is the 95% CI propagated through the covariance of β₂ and β₃. This is the key inferential figure: if the line slopes downward from left (drought) to right (wet), β₃ < 0 and irrigation is relatively *more* beneficial under drought.


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── LEFT PANEL: coefficient bar chart with 95% CI ──────────────────────────────────────────────────
ax    = axes[0]
coefs = [b1, b2, b3]
lo    = [res.conf_int().loc[v, 0] for v in key_vars]
hi    = [res.conf_int().loc[v, 1] for v in key_vars]
xlabs = ['SPEI-90d\n(drought)', '\u0394ET\n(irrigation)', 'SPEI \u00d7 \u0394ET\n(interaction)']
clrs  = ['#2196F3', '#4CAF50', '#FF5722']
xs    = np.arange(len(coefs))

ax.bar(xs, coefs, color=clrs, alpha=0.8, edgecolor='black', linewidth=0.8, width=0.5)
ax.errorbar(xs, coefs,
            yerr=[[c - l for c, l in zip(coefs, lo)],
                  [h - c for c, h in zip(coefs, hi)]],
            fmt='none', color='black', capsize=7, linewidth=1.5)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.6)
ax.set_xticks(xs)
ax.set_xticklabels(xlabs, fontsize=11)
ax.set_ylabel('Regression coefficient (SIF z-score units)', fontsize=10)
ax.set_title('SIF Z-score Regression Coefficients\nPooled OLS + month FE, HC3 SE', fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Add significance stars above/below each bar
for xi, (coef, var, lhi, llo) in enumerate(zip(coefs, key_vars, hi, lo)):
    p    = res.pvalues[var]
    star = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
    # Position star just outside the CI
    ypos = lhi + abs(lhi - coef) * 0.15 if coef >= 0 else llo - abs(llo - coef) * 0.15
    va   = 'bottom' if coef >= 0 else 'top'
    ax.text(xi, ypos, star, ha='center', va=va, fontsize=12, fontweight='bold')

# ── RIGHT PANEL: marginal effect of ΔET vs SPEI ──────────────────────────────────────────────────
ax = axes[1]

# Compute ME = b2 + b3*SPEI across the SPEI range, with propagated 95% CI
# Var(ME) = Var(b2) + SPEI^2 * Var(b3) + 2*SPEI * Cov(b2, b3)
spei_range = np.linspace(-3, 2, 400)
me_range   = b2 + b3 * spei_range
cov_b2_b3  = res.cov_params().loc['delta_et', 'spei_x_det']
se_me = np.sqrt(
    res.bse['delta_et']**2
    + spei_range**2 * res.bse['spei_x_det']**2
    + 2 * spei_range * cov_b2_b3
)

# Shade drought zone (SPEI < 0)
ax.axvspan(-3, 0, alpha=0.05, color='red')

# 95% confidence band
ax.fill_between(spei_range,
                me_range - 1.96 * se_me,
                me_range + 1.96 * se_me,
                alpha=0.25, color='#4CAF50', label='95% CI')

# Central estimate line
ax.plot(spei_range, me_range, color='#1b5e20', linewidth=2.5,
        label='\u2202SIF_z/\u2202\u0394ET = b\u2082 + b\u2083\u00b7SPEI')

ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.6)
ax.axvline(0, color='gray',  linewidth=0.8, linestyle=':', alpha=0.7)

ax.set_xlim(-3, 2)
ax.set_xlabel('SPEI-90d  (drought \u2190  0  \u2192 wet)', fontsize=11)
ax.set_ylabel('Marginal effect of \u0394ET on SIF z-score\n(SIF z-score per mm/mo HumanET)', fontsize=10)
ax.set_title('How Irrigation\u2019s Effect on SIF Varies with Drought Severity', fontsize=11)
ax.text(0.04, 0.92, 'Drought zone (SPEI < 0)',
        transform=ax.transAxes, color='red', fontsize=9, alpha=0.8)
ax.legend(fontsize=9, loc='lower right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(figs / 'reg_coefficients_marginal_effect.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reg_coefficients_marginal_effect.png')


Saved: reg_coefficients_marginal_effect.png


## 7. Spatial Analysis: Where Does Irrigation Buffer SIF Under Drought?

The pooled regression gives a single CONUS-wide estimate of β₃. To understand the *spatial heterogeneity* of the irrigation buffering effect, we run pixel-level simple regressions of SIF_z on ΔET, restricted to drought months only (SPEI ≤ −0.5).

**Method:**
1. Filter to observations where SPEI ≤ −0.5 (mild drought or worse).
2. For each cropland pixel with ≥ 5 drought-month observations, regress `sif_z ~ delta_et`.
3. Retain pixels where the slope is significant at p < 0.10 (liberal threshold for a spatial exploration).
4. Map the slope coefficients back to the CONUS grid.

**Interpretation of the map:**
- **Blue (positive slope):** In this pixel, more HumanET is associated with higher SIF z-scores during drought months — consistent with an irrigation buffering effect.
- **Red (negative slope):** More HumanET during drought is associated with *lower* SIF z-scores. This can occur if irrigated crops in severe water-stress regions still struggle, or if the ΔET signal captures non-irrigation processes.
- **Gray (no color):** Pixel was not sampled, had insufficient drought-month observations, or the slope was not significant at p < 0.10.

Note that this pixel-level approach is descriptive and does not control for confounders; the causal estimate comes from the pooled model in Section 5.


In [8]:
# ── Filter to drought months only ─────────────────────────────────────────────────────────────────────────────────
df_drought = df_reg[df_reg['spei90d'] <= DROUGHT_SPEI_THRESH].copy()
print('Drought observations (SPEI \u2264 ' + str(DROUGHT_SPEI_THRESH) + '): ' + f'{len(df_drought):,}')
print('Unique pixels in drought subset: ' + str(df_drought.groupby(['lat','lon']).ngroups))
print()

# ── Fit pixel-level regressions: SIF_z ~ ΔET (drought months only) ─────────────────────────────────────
def _pixel_slope(grp):
    """Return OLS slope and p-value for SIF_z ~ delta_et within one pixel."""
    xy = grp[['delta_et', 'sif_z']].dropna()
    if len(xy) < MIN_OBS_PIXEL:
        return pd.Series({'slope': np.nan, 'pval': np.nan, 'n': len(xy)})
    slope, intercept, r, pval, se = stats.linregress(xy['delta_et'].values,
                                                      xy['sif_z'].values)
    return pd.Series({'slope': slope, 'pval': pval, 'n': len(xy)})

# Group by pixel and apply regression
# This may take 30–120 seconds for ~25K cropland pixels × drought months.
print('Running pixel-level regressions...')
pix_stats = (df_drought
             .groupby(['lat', 'lon'])
             .apply(_pixel_slope)
             .reset_index())

# Keep only pixels with significant slopes (p < ALPHA_SPATIAL)
pix_sig = pix_stats[pix_stats['pval'] < ALPHA_SPATIAL].copy()

print('Pixels with >= ' + str(MIN_OBS_PIXEL) + ' drought obs: ' + str(pix_stats['slope'].notna().sum()))
print('Significant at p < ' + str(ALPHA_SPATIAL) + ': ' + str(len(pix_sig)))
print()

# Direction breakdown
n_pos = (pix_sig['slope'] > 0).sum()
n_neg = (pix_sig['slope'] < 0).sum()
print('Positive slopes (irrigation buffers SIF): ' + str(n_pos) +
      ' (' + f'{100*n_pos/len(pix_sig):.1f}' + '%)')
print('Negative slopes (no buffering):           ' + str(n_neg) +
      ' (' + f'{100*n_neg/len(pix_sig):.1f}' + '%)')


Drought observations (SPEI ≤ -0.5): 120,766
Unique pixels in drought subset: 7571

Running pixel-level regressions...


Pixels with >= 5 drought obs: 7456
Significant at p < 0.1: 2032

Positive slopes (irrigation buffers SIF): 1701 (83.7%)
Negative slopes (no buffering):           331 (16.3%)


In [9]:
# ── Map pixel slopes onto the CONUS grid ───────────────────────────────────────────────────────────────────────
slope_map = np.full((n_lat, n_lon), np.nan)

for _, row in pix_sig.iterrows():
    # Find the nearest grid cell for this pixel's lat/lon
    ri = np.argmin(np.abs(CONUS_LAT - row['lat']))
    ci = np.argmin(np.abs(CONUS_LON - row['lon']))
    slope_map[ri, ci] = row['slope']

# Mask to cropland if available
if crop_mask_static is not None:
    slope_map[~crop_mask_static] = np.nan

# ── Color scale: symmetric around 0, clipped at 95th percentile of |slope| ────────────────────
finite_vals = slope_map[np.isfinite(slope_map)]
if len(finite_vals) == 0:
    print('No significant pixel slopes to map. Try loosening MIN_OBS_PIXEL or ALPHA_SPATIAL.')
else:
    vmax    = max(np.nanpercentile(np.abs(finite_vals), 95), 0.001)
    pct_pos = 100.0 * (finite_vals > 0).mean()

    fig, ax = plt.subplots(figsize=(17, 7))

    im = ax.imshow(
        slope_map,
        extent=[CONUS_LON[0], CONUS_LON[-1], CONUS_LAT[-1], CONUS_LAT[0]],
        origin='upper',
        cmap='RdBu',        # Red = negative slope, Blue = positive (buffering)
        vmin=-vmax,
        vmax=vmax,
        aspect='auto'
    )

    cb = plt.colorbar(im, ax=ax, fraction=0.022, pad=0.02)
    cb.set_label(
        '\u0394ET \u2192 SIF z-score slope during drought (SPEI \u2264 '
        + str(DROUGHT_SPEI_THRESH)
        + ', p < '
        + str(ALPHA_SPATIAL)
        + ')\n'
        'Blue = irrigation associated with higher SIF (buffering)  '
        '\u2502  Red = no buffering benefit',
        fontsize=10
    )

    ax.set_title(
        'Spatial Irrigation Buffering Effect on SIF Under Drought \u2014 CONUS Cropland\n'
        + f'{pct_pos:.0f}% of significant pixels show positive buffering (blue)',
        fontsize=13
    )
    ax.set_xlabel('Longitude', fontsize=11)
    ax.set_ylabel('Latitude', fontsize=11)

    plt.tight_layout()
    plt.savefig(figs / 'reg_sif_irrigation_buffer_spatial.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('Map statistics:')
    print('  Total significant pixels mapped: ' + str(len(finite_vals)))
    print('  Positive (buffering): ' + str((finite_vals > 0).sum()))
    print('  Negative:             ' + str((finite_vals < 0).sum()))
    print('  Color range: \u00b1' + f'{vmax:.4f}')
    print('Saved: reg_sif_irrigation_buffer_spatial.png')


Map statistics:
  Total significant pixels mapped: 2032
  Positive (buffering): 1701
  Negative:             331
  Color range: ±0.0546
Saved: reg_sif_irrigation_buffer_spatial.png


## 8. Summary & Interpretation

### What the regression tells us

The pooled OLS model with month fixed effects estimates the interaction between drought (SPEI-90d) and irrigation intensity (HumanET / ΔET) on SIF anomalies across CONUS cropland.

**β₁ (SPEI):** The main effect of drought on SIF — we expect this to be positive (higher SPEI = wetter = higher SIF). A significant positive coefficient confirms that the SIF z-score tracks the drought gradient, validating the dataset.

**β₂ (ΔET):** The main effect of irrigation on SIF anomaly under average drought conditions (SPEI = 0). A positive value means that higher HumanET is associated with above-normal SIF even in non-drought months, consistent with irrigation supporting crop growth.

**β₃ (SPEI × ΔET):** The key interaction. The marginal effect curve (Figure 2) shows how β₂ changes across the drought gradient. If negative: the irrigation-SIF slope is steeper when drought is severe, meaning irrigation has an outsized positive effect on SIF exactly when drought stress is greatest — the buffering hypothesis.

### Spatial patterns

The pixel-level map (Section 7) reveals where buffering is strongest. Expected patterns (if the hypothesis holds):
- **High Plains / Nebraska Sandhills:** Dense center-pivot irrigation, consistent groundwater access → strong buffering.
- **California Central Valley:** Extensive surface-water irrigation → buffering where water available.
- **Corn Belt:** Less irrigation overall, but buffering detectable in drought years.
- **Southeast / dryland Southeast:** Minimal irrigation → negative or null slopes expected.

### Limitations

1. **20% pixel sample:** Spatial maps will be sparse; full-pixel analysis would improve coverage.
2. **Causality:** ΔET is not randomly assigned — irrigated farms self-select into locations with water access. The interaction estimate is associational, not strictly causal.
3. **SIF z-score scale:** A one-unit change in SIF z-score represents one standard deviation from the local mean — magnitudes are unitless and not directly comparable to yield impacts.
4. **Panel FE:** A pixel fixed-effects estimator (e.g., `linearmodels.PanelOLS`) is preferred for unbiased coefficients if irrigated pixels differ systematically in time-invariant ways (soil quality, crop mix). This is a natural extension.


In [10]:
# ── Print a concise results summary ─────────────────────────────────────────────────────────────────────────────
print('=' * 65)
print('SIF REGRESSION SUMMARY — CONUS CROPLAND 2015-2024')
print('=' * 65)
print()
print('Model: SIF_z ~ SPEI + \u0394ET + SPEI\u00d7\u0394ET + C(month)')
print('       Pooled OLS, HC3 robust SE')
print()
print(f'  N observations : {int(res.nobs):,}')
print(f'  R\u00b2             : {res.rsquared:.4f}')
print(f'  Adj. R\u00b2        : {res.rsquared_adj:.4f}')
print()
print('Key coefficients:')
for var, lbl in zip(key_vars, row_labels):
    coef = res.params[var]
    se   = res.bse[var]
    pval = res.pvalues[var]
    sig  = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else 'n.s.'))
    print('  ' + lbl + ':')
    print('    \u03b2 = ' + f'{coef:.5f}' + '  SE = ' + f'{se:.5f}' +
          '  p = ' + f'{pval:.4g}' + '  ' + sig)
print()
print('Marginal effect of \u0394ET at SPEI = -1.5 (moderate drought):')
me_d2 = b2 + b3 * (-1.5)
print('  ' + f'{me_d2:+.5f}' + ' SIF z-score per mm/mo HumanET')
print()

if 'finite_vals' in dir() and len(finite_vals) > 0:
    print('Spatial analysis (pixel slopes, drought months, p < ' + str(ALPHA_SPATIAL) + '):')
    print('  Significant pixels: ' + str(len(finite_vals)))
    print('  Positive (buffering): ' + f'{100*(finite_vals>0).mean():.1f}' + '%')
    print('  Negative:             ' + f'{100*(finite_vals<0).mean():.1f}' + '%')
print()
print('Figures saved to:', figs)


SIF REGRESSION SUMMARY — CONUS CROPLAND 2015-2024

Model: SIF_z ~ SPEI + ΔET + SPEI×ΔET + C(month)
       Pooled OLS, HC3 robust SE

  N observations : 434,245
  R²             : 0.2055
  Adj. R²        : 0.2055

Key coefficients:
  SPEI-90d (drought):
    β = 0.17264  SE = 0.00209  p = 0  ***
  ΔET / HumanET:
    β = 0.00357  SE = 0.00006  p = 0  ***
  SPEI × ΔET (interaction):
    β = 0.00067  SE = 0.00006  p = 8.097e-26  ***

Marginal effect of ΔET at SPEI = -1.5 (moderate drought):
  +0.00257 SIF z-score per mm/mo HumanET

Spatial analysis (pixel slopes, drought months, p < 0.1):
  Significant pixels: 2032
  Positive (buffering): 83.7%
  Negative:             16.3%

Figures saved to: /home/pielab-sandbox-jcoldiron/SIF-Analysis/figures/conus


---

## 9. Expanded Regression with NLDAS Noah Covariates

This section tests an expanded regression model by adding six additional climate/land-surface variables available directly from the NLDAS-2 Noah LSM output files (`data/raw/nldas/`). These variables capture aspects of the surface energy balance, soil moisture state, and precipitation that may explain SIF anomalies beyond what SPEI and ΔET alone can account for.

**Additional predictors loaded from NLDAS Noah files:**

| Variable | Description | Units |
|---|---|---|
| `SWdown` | Downwelling shortwave radiation | W m⁻² |
| `AvgSurfT` | Average surface skin temperature | K |
| `PotEvap` | Potential evapotranspiration | W m⁻² |
| `SoilM_0_10cm` | Shallow soil moisture content (0–10 cm) | kg m⁻² |
| `Rainf` | Liquid precipitation (monthly total) | kg m⁻² |
| `ACond` | Aerodynamic conductance | m s⁻¹ |

**Workflow:**
1. Load these variables from raw NLDAS files for growing-season months (April–September), 2015–2024
2. Extract cropland pixels using `crop_mask_static`; align to `df_reg` by lat, lon, year, month
3. Compute a Pearson correlation matrix across all eight candidate predictors; flag |r| > 0.70
4. Fit an expanded OLS regression with HC3 robust SEs; compare R² to the baseline (0.2055)

### 9.1 Load and Align NLDAS Variables


In [11]:

import xarray as xr

_NLDAS_VARS = ['SWdown', 'AvgSurfT', 'PotEvap', 'SoilM_0_10cm', 'Rainf', 'ACond']
_nldas_dir  = project_root / 'data' / 'raw' / 'nldas'

# ── Identify cropland pixel positions in the CONUS grid ───────────────────────
if crop_mask_static is None:
    raise RuntimeError('crop_mask_static is required to extract NLDAS cropland pixels')

_crop_rows, _crop_cols = np.where(crop_mask_static)  # row/col indices into CONUS grid
_crop_lats = CONUS_LAT[_crop_rows]
_crop_lons = CONUS_LON[_crop_cols]
print('Cropland pixels in CONUS mask:', len(_crop_lats))

# ── Probe one file to confirm lat/lon grid ─────────────────────────────────────
_probe_path = sorted(_nldas_dir.glob('NLDAS_NOAH0125_M.A2019*.nc'))[0]
with xr.open_dataset(_probe_path) as _ds_probe:
    _nldas_lat = _ds_probe['lat'].values
    _nldas_lon = _ds_probe['lon'].values
print('NLDAS lat: ' + str(round(float(_nldas_lat.min()), 4)) +
      ' to ' + str(round(float(_nldas_lat.max()), 4)) +
      ' (' + str(len(_nldas_lat)) + ' values)')
print('NLDAS lon: ' + str(round(float(_nldas_lon.min()), 4)) +
      ' to ' + str(round(float(_nldas_lon.max()), 4)) +
      ' (' + str(len(_nldas_lon)) + ' values)')
print()

# ── Load growing-season months 2015–2024 ──────────────────────────────────────
_all_dfs = []
for year in YEARS:
    for month in GROWING_SEASON:
        _fname = ('NLDAS_NOAH0125_M.A'
                  + str(year)
                  + str(month).zfill(2)
                  + '.020.nc')
        _fpath = _nldas_dir / _fname
        if not _fpath.exists():
            print('  WARNING: missing file:', _fname)
            continue

        with xr.open_dataset(_fpath) as _ds:
            # Select CONUS subdomain — grids share 0.125° spacing so nearest=exact
            _ds_sub = _ds[_NLDAS_VARS].sel(
                lat=CONUS_LAT, lon=CONUS_LON, method='nearest'
            )

            _mdf = pd.DataFrame({
                'year':  year,
                'month': month,
                'lat':   _crop_lats,
                'lon':   _crop_lons,
            })
            for _v in _NLDAS_VARS:
                _arr = _ds_sub[_v].values.squeeze()        # (189, 325)
                _mdf[_v] = _arr[_crop_rows, _crop_cols]    # (n_cropland,)

            _all_dfs.append(_mdf)

    print('  Loaded year', year)

df_nldas_new = pd.concat(_all_dfs, ignore_index=True)
print()
print('NLDAS extract shape:', df_nldas_new.shape)
print()
for _v in _NLDAS_VARS:
    print(_v + ':  min=' + f'{df_nldas_new[_v].min():.3g}' +
          '  mean=' + f'{df_nldas_new[_v].mean():.3g}' +
          '  max=' + f'{df_nldas_new[_v].max():.3g}')

# ── Merge into the regression dataset (df_reg) ────────────────────────────────
# Round lat/lon to 4 decimal places to neutralise floating-point drift
df_nldas_new['lat_r'] = df_nldas_new['lat'].round(4)
df_nldas_new['lon_r'] = df_nldas_new['lon'].round(4)

_df_reg_work = df_reg.copy()
_df_reg_work['lat_r'] = _df_reg_work['lat'].round(4)
_df_reg_work['lon_r'] = _df_reg_work['lon'].round(4)

df_expanded = _df_reg_work.merge(
    df_nldas_new[['lat_r', 'lon_r', 'year', 'month'] + _NLDAS_VARS],
    on=['lat_r', 'lon_r', 'year', 'month'],
    how='left',
)

join_rate = df_expanded[_NLDAS_VARS[0]].notna().sum()
print()
print('df_expanded shape: ' + str(df_expanded.shape))
print('Rows with NLDAS values joined: ' + f'{join_rate:,}' +
      ' / ' + f'{len(df_expanded):,}' +
      ' (' + f'{100*join_rate/len(df_expanded):.1f}' + '%)')


Cropland pixels in CONUS mask: 25153


NLDAS lat: 25.0625 to 52.9375 (224 values)
NLDAS lon: -124.9375 to -67.0625 (464 values)



  Loaded year 2015


  Loaded year 2016


  Loaded year 2017


...

  Loaded year 2021


  Loaded year 2022


  Loaded year 2023


  Loaded year 2024

NLDAS extract shape: (1509180, 10)

SWdown:  min=123  mean=263  max=383
AvgSurfT:  min=261  mean=294  max=313
PotEvap:  min=3.76  mean=249  max=688
SoilM_0_10cm:  min=2  mean=20.9  max=46.4
Rainf:  min=0  mean=64.9  max=677
ACond:  min=0.000464  mean=0.019  max=0.0481



df_expanded shape: (434245, 22)
Rows with NLDAS values joined: 434,245 / 434,245 (100.0%)


### 9.2 Predictor Correlation Matrix

We compute the Pearson correlation matrix across all eight candidate predictors using complete cases from `df_expanded`. Pairs with |r| > 0.70 are flagged as potentially collinear — these can inflate standard errors in the expanded model without necessarily biasing coefficients.


In [12]:

# ── Pearson correlation matrix across all candidate predictors ─────────────────
PRED_COLS = ['spei90d', 'delta_et', 'SWdown', 'AvgSurfT', 'PotEvap',
             'SoilM_0_10cm', 'Rainf', 'ACond']

df_corr_input = df_expanded[PRED_COLS].dropna()
print('Complete cases for correlation matrix: ' + f'{len(df_corr_input):,}')
print()

corr_mat = df_corr_input.corr(method='pearson')

print('Pearson correlation matrix:')
print(corr_mat.round(3).to_string())
print()

# Flag highly collinear pairs
HIGH_R = 0.70
print('Pairs with |r| > ' + str(HIGH_R) + ':')
found_high = False
for i, c1 in enumerate(PRED_COLS):
    for j, c2 in enumerate(PRED_COLS):
        if j <= i:
            continue
        r = corr_mat.loc[c1, c2]
        if abs(r) > HIGH_R:
            print('  ' + c1 + ' — ' + c2 + ':  r = ' + f'{r:.3f}')
            found_high = True
if not found_high:
    print('  (none above ' + str(HIGH_R) + ' threshold)')

# ── Heatmap ────────────────────────────────────────────────────────────────────
try:
    import seaborn as sns
    _use_sns = True
except ImportError:
    _use_sns = False

fig, ax = plt.subplots(figsize=(9, 7))

if _use_sns:
    sns.heatmap(
        corr_mat,
        ax=ax,
        annot=True,
        fmt='.2f',
        cmap='RdBu_r',
        vmin=-1, vmax=1,
        linewidths=0.5,
        annot_kws={'size': 9},
        square=True,
        cbar_kws={'shrink': 0.8, 'label': 'Pearson r'},
    )
else:
    im = ax.imshow(corr_mat.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    cb = plt.colorbar(im, ax=ax, shrink=0.8)
    cb.set_label('Pearson r')
    ax.set_xticks(range(len(PRED_COLS)))
    ax.set_xticklabels(PRED_COLS, rotation=30, ha='right', fontsize=9)
    ax.set_yticks(range(len(PRED_COLS)))
    ax.set_yticklabels(PRED_COLS, fontsize=9)
    for i in range(len(PRED_COLS)):
        for j in range(len(PRED_COLS)):
            ax.text(j, i, f'{corr_mat.iloc[i, j]:.2f}',
                    ha='center', va='center', fontsize=8)

# Outline cells with |r| > HIGH_R (excluding diagonal)
for i in range(len(PRED_COLS)):
    for j in range(len(PRED_COLS)):
        if i != j and abs(corr_mat.iloc[i, j]) > HIGH_R:
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5) if not _use_sns else (j, i),
                                        1, 1, fill=False, edgecolor='black', lw=2.5))

ax.set_title('Predictor Correlation Matrix  (black box = |r| > ' + str(HIGH_R) + ')', fontsize=12)
ax.tick_params(axis='x', rotation=30)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(figs / 'predictor_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: predictor_correlation_matrix.png')


Complete cases for correlation matrix: 434,245

Pearson correlation matrix:
              spei90d  delta_et  SWdown  AvgSurfT  PotEvap  SoilM_0_10cm  Rainf  ACond
spei90d         1.000    -0.156  -0.058     0.009   -0.209         0.329  0.491 -0.052
delta_et       -0.156     1.000   0.263     0.176    0.499        -0.529 -0.380  0.214
SWdown         -0.058     0.263   1.000     0.619    0.736        -0.190 -0.052  0.056
AvgSurfT        0.009     0.176   0.619     1.000    0.695        -0.243  0.102 -0.238
PotEvap        -0.209     0.499   0.736     0.695    1.000        -0.532 -0.267  0.259
SoilM_0_10cm    0.329    -0.529  -0.190    -0.243   -0.532         1.000  0.581  0.022
Rainf           0.491    -0.380  -0.052     0.102   -0.267         0.581  1.000 -0.015
ACond          -0.052     0.214   0.056    -0.238    0.259         0.022 -0.015  1.000

Pairs with |r| > 0.7:
  SWdown — PotEvap:  r = 0.736


Saved: predictor_correlation_matrix.png


### 9.3 Expanded OLS Regression

We fit the expanded model adding all six NLDAS surface variables alongside the baseline predictors:

$$\text{SIF}_z = \beta_1 \text{SPEI} + \beta_2 \Delta\text{ET} + \beta_3 (\text{SPEI} \times \Delta\text{ET}) + \beta_4 \text{SWdown} + \beta_5 T_\text{surf} + \beta_6 \text{PotEvap} + \beta_7 \text{SoilM} + \beta_8 \text{Rainf} + \beta_9 \text{ACond} + \alpha_m + \varepsilon$$

HC3 robust standard errors are used throughout, matching the baseline model specification. The sample is restricted to complete cases across all nine predictors plus `sif_z`.


In [13]:

# ── Expanded OLS: SIF_z ~ SPEI + ΔET + SPEI×ΔET + NLDAS covariates + C(month) ─
NLDAS_VARS_NEW = ['SWdown', 'AvgSurfT', 'PotEvap', 'SoilM_0_10cm', 'Rainf', 'ACond']

REG_COLS_EXP = ['sif_z', 'spei90d', 'delta_et', 'spei_x_det', 'month'] + NLDAS_VARS_NEW
df_exp_reg = df_expanded[REG_COLS_EXP].dropna().copy()
df_exp_reg = df_exp_reg[
    np.isfinite(df_exp_reg['sif_z']) &
    np.isfinite(df_exp_reg['spei90d']) &
    np.isfinite(df_exp_reg['delta_et'])
].copy()

print('Expanded regression dataset: ' + f'{len(df_exp_reg):,}' + ' observations')
print('Baseline had: ' + f'{int(res.nobs):,}' + ' observations')
print()

formula_expanded = (
    'sif_z ~ spei90d + delta_et + spei_x_det'
    ' + SWdown + AvgSurfT + PotEvap + SoilM_0_10cm + Rainf + ACond'
    ' + C(month)'
)

res_exp = smf.ols(formula_expanded, data=df_exp_reg).fit(cov_type='HC3')

# ── Print clean coefficient table ─────────────────────────────────────────────
main_vars = ['spei90d', 'delta_et', 'spei_x_det'] + NLDAS_VARS_NEW
var_labels = {
    'spei90d':       'SPEI-90d',
    'delta_et':      'ΔET (HumanET)',
    'spei_x_det':    'SPEI × ΔET',
    'SWdown':        'SWdown (W m-2)',
    'AvgSurfT':      'AvgSurfT (K)',
    'PotEvap':       'PotEvap (W m-2)',
    'SoilM_0_10cm':  'SoilM 0-10cm (kg m-2)',
    'Rainf':         'Rainf (kg m-2)',
    'ACond':         'ACond (m s-1)',
}

print('Expanded OLS Results — HC3 Robust Standard Errors')
print('Model: SIF_z ~ SPEI + ΔET + SPEI×ΔET + SWdown + AvgSurfT + PotEvap')
print('       + SoilM_0_10cm + Rainf + ACond + C(month)')
print()
print('N = ' + f'{int(res_exp.nobs):,}' +
      '   R² = ' + f'{res_exp.rsquared:.4f}' +
      '   Adj. R² = ' + f'{res_exp.rsquared_adj:.4f}')
print()
print(f'{"Variable":<26} {"Coef.":>10} {"Std.Err.":>10} {"p-value":>11}  Sig.')
print('-' * 66)
for v in main_vars:
    lbl  = var_labels.get(v, v)
    coef = res_exp.params[v]
    se   = res_exp.bse[v]
    pval = res_exp.pvalues[v]
    sig  = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else 'n.s.'))
    print(f'  {lbl:<24} {coef:>10.5f} {se:>10.5f} {pval:>11.4g}  {sig}')

print()
print('Month fixed effects (relative to reference):')
for k, v in sorted(res_exp.params.items()):
    if k.startswith('C(month)'):
        m = k.split('[T.')[-1].rstrip(']')
        p = res_exp.pvalues[k]
        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
        print('  Month ' + m + ':  ' + f'{v:+.4f}' + '  ' + sig)


Expanded regression dataset: 434,245 observations
Baseline had: 434,245 observations



Expanded OLS Results — HC3 Robust Standard Errors
Model: SIF_z ~ SPEI + ΔET + SPEI×ΔET + SWdown + AvgSurfT + PotEvap
       + SoilM_0_10cm + Rainf + ACond + C(month)

N = 434,245   R² = 0.2119   Adj. R² = 0.2119

Variable                        Coef.   Std.Err.     p-value  Sig.
------------------------------------------------------------------
  SPEI-90d                    0.17711    0.00241           0  ***
  ΔET (HumanET)               0.00368    0.00007           0  ***
  SPEI × ΔET                  0.00057    0.00007   5.986e-18  ***
  SWdown (W m-2)              0.00229    0.00010  2.018e-116  ***
  AvgSurfT (K)               -0.01020    0.00060   4.097e-65  ***
  PotEvap (W m-2)             0.00125    0.00007   1.249e-79  ***
  SoilM 0-10cm (kg m-2)       0.01137    0.00037  1.699e-204  ***
  Rainf (kg m-2)             -0.00011    0.00004    0.003444  **
  ACond (m s-1)             -15.31287    0.48258  5.814e-221  ***

Month fixed effects (relative to reference):
  Month 5:  -0

In [14]:

# ── Side-by-side model comparison ─────────────────────────────────────────────
r2_base = res.rsquared
r2_exp  = res_exp.rsquared
delta_r2 = r2_exp - r2_base
pct_gain = 100.0 * delta_r2 / r2_base if r2_base > 0 else float('nan')

print('=' * 60)
print('MODEL COMPARISON: BASELINE vs EXPANDED')
print('=' * 60)
print()
print(f'{"":32s} {"Baseline":>10} {"Expanded":>10}')
print('-' * 56)
print(f'{"Observations":32s} {int(res.nobs):>10,} {int(res_exp.nobs):>10,}')
print(f'{"Non-month predictors":32s} {"3":>10} {"9":>10}')
print(f'{"R²":32s} {r2_base:>10.4f} {r2_exp:>10.4f}')
print(f'{"Adj. R²":32s} {res.rsquared_adj:>10.4f} {res_exp.rsquared_adj:>10.4f}')
print(f'{"AIC":32s} {res.aic:>10.1f} {res_exp.aic:>10.1f}')
print(f'{"BIC":32s} {res.bic:>10.1f} {res_exp.bic:>10.1f}')
print('-' * 56)
print(f'{"ΔR² (expanded − baseline)":32s} {delta_r2:>10.4f}')
print()
print('R² gain: ' + f'{delta_r2:.4f}' + '  (' + f'{pct_gain:.1f}' + '% relative improvement over baseline)')
print()

if r2_exp > r2_base + 0.01:
    print('Interpretation: The expanded model meaningfully improves fit.')
    print('The NLDAS surface covariates capture additional SIF variance')
    print('beyond SPEI and ΔET alone.')
elif r2_exp > r2_base:
    print('Interpretation: Modest improvement. The NLDAS covariates add')
    print('incremental explanatory power; the core SPEI + ΔET terms dominate.')
else:
    print('Interpretation: No improvement over baseline.')
    print('Check multicollinearity or reduced sample size after the NLDAS join.')


MODEL COMPARISON: BASELINE vs EXPANDED

                                   Baseline   Expanded
--------------------------------------------------------
Observations                        434,245    434,245
Non-month predictors                      3          9
R²                                   0.2055     0.2119
Adj. R²                              0.2055     0.2119
AIC                               1110990.5  1107524.2
BIC                               1111089.3  1107688.9
--------------------------------------------------------
ΔR² (expanded − baseline)            0.0063

R² gain: 0.0063  (3.1% relative improvement over baseline)

Interpretation: Modest improvement. The NLDAS covariates add
incremental explanatory power; the core SPEI + ΔET terms dominate.


---

## 10. Adding GRIDMET VPD to the Expanded Regression

VPD (vapor pressure deficit) is a key driver of plant water stress and is expected to co-vary strongly with SIF anomalies, particularly under drought. This section attempts to add monthly GRIDMET VPD as an additional predictor.

**Strategy:** First inspect the processed GRIDMET files at `data/processed/conus/drought_gridmet/` for a VPD band. If absent, fall back to the raw files at `data/raw/drought_gridmet/`. If found, extract cropland pixels, merge into `df_expanded`, update the correlation matrix (paying particular attention to VPD–PotEvap collinearity), and run the extended model. If not found, document exactly what is present and stop.


In [15]:

import rasterio

_gm_proc_dir = project_root / 'data' / 'processed' / 'conus' / 'drought_gridmet'
_gm_raw_dir  = project_root / 'data' / 'raw' / 'drought_gridmet'

_PROBE_YYYYMM = '201907'
_proc_file = _gm_proc_dir / ('GRIDMET_drought_' + _PROBE_YYYYMM + '.tif')
_raw_file  = _gm_raw_dir  / ('GRIDMET_drought_' + _PROBE_YYYYMM + '.tif')

# ── Helper: describe one GeoTIFF ──────────────────────────────────────────────
def _describe_tif(path, label):
    print('--- ' + label + ': ' + str(path))
    if not path.exists():
        print('  NOT FOUND')
        return {}
    band_info = {}
    with rasterio.open(path) as src:
        print('  CRS:    ' + str(src.crs))
        print('  Shape:  ' + str(src.height) + ' rows x ' + str(src.width) + ' cols')
        print('  Bands:  ' + str(src.count))
        print('  Nodata: ' + str(src.nodata))
        print()
        print(f'  {"Band":>5}  {"description":>20}  {"tags":>30}  '
              f'{"min":>8}  {"mean":>8}  {"max":>8}')
        print('  ' + '-' * 85)
        for b in range(1, src.count + 1):
            desc   = src.descriptions[b - 1] or '(none)'
            btags  = src.tags(b)
            bname  = btags.get('name', btags.get('long_name', '(none)'))
            arr    = src.read(b).astype(float)
            nd     = src.nodata
            if nd is not None:
                arr[arr == nd] = np.nan
            vmin   = np.nanmin(arr)
            vmean  = np.nanmean(arr)
            vmax   = np.nanmax(arr)
            print(f'  {b:>5}  {desc:>20}  {bname:>30}  '
                  f'{vmin:>8.3f}  {vmean:>8.3f}  {vmax:>8.3f}')
            band_info[b] = {'desc': desc, 'name': bname,
                            'min': vmin, 'mean': vmean, 'max': vmax}
    return band_info

print('=' * 70)
print('SECTION 10 — GRIDMET FILE INSPECTION FOR VPD')
print('Representative month: ' + _PROBE_YYYYMM)
print('=' * 70)
print()

proc_bands = _describe_tif(_proc_file, 'PROCESSED file')
print()
raw_bands  = _describe_tif(_raw_file,  'RAW file')

# ── Search every band name for VPD ───────────────────────────────────────────
print()
print('=' * 70)
print('VPD SEARCH RESULT')
print('=' * 70)

_vpd_keywords = ['vpd', 'vapor', 'deficit']
_all_bands = list(proc_bands.values()) + list(raw_bands.values())
_found_vpd = [b for b in _all_bands
              if any(kw in str(b.get('name', '')).lower() or
                     kw in str(b.get('desc', '')).lower()
                     for kw in _vpd_keywords)]

if _found_vpd:
    print('VPD found in: ' + str(_found_vpd))
else:
    print()
    print('VPD is NOT present in any band of either the processed or raw')
    print('GRIDMET drought files.')
    print()
    print('What IS in these files (8 bands, no metadata tags):')
    print()
    print('  Bands 1-6 — values clipped near ±2.09 → SPEI/SPI at multiple timescales')
    print('    (notebook 01 uses band 1 as SPEI-90d; the ±2.09 cap is the SPEI')
    print('     truncation applied during index computation)')
    print()
    print('  Band 7   — range roughly -5.5 to +11  → likely PDSI')
    print('    (PDSI is unbounded, positive = wet; range matches GRIDMET PDSI product)')
    print()
    print('  Band 8   — range roughly -5.0 to +11  → likely EDDI (z-score form)')
    print('    (Evaporative Demand Drought Index; positive = anomalously high demand)')
    print()
    print('These files are the GRIDMET drought indices product (from climate-engine /')
    print('GRIDMET via GEE), which bundles SPEI, SPI, PDSI, and EDDI — not the')
    print('raw GRIDMET climate variables (tmax, tmin, vpd, pr, etc.).')
    print()
    print('To add VPD you would need to separately download the GRIDMET climate')
    print('variable "vpd" (mean daily VPD in kPa, monthly aggregated), e.g. via:')
    print('  https://www.climatologylab.org/gridmet.html  or')
    print('  Google Earth Engine: ee.ImageCollection("IDAHO_EPSCOR/GRIDMET")')
    print('    .select("vpd")  -- units: kPa, ~0.01 to 4+ kPa over CONUS')
    print()
    print('STOPPING — no VPD data available to merge. Expanded + VPD regression')
    print('cannot be run until VPD files are downloaded and aligned to the CONUS grid.')


SECTION 10 — GRIDMET FILE INSPECTION FOR VPD
Representative month: 201907

--- PROCESSED file: /home/pielab-sandbox-jcoldiron/SIF-Analysis/data/processed/conus/drought_gridmet/GRIDMET_drought_201907.tif
  CRS:    EPSG:4326
  Shape:  189 rows x 325 cols
  Bands:  8
  Nodata: None

   Band           description                            tags       min      mean       max
  -------------------------------------------------------------------------------------
      1                (none)                          (none)    -2.090     0.075     2.090
      2                (none)                          (none)    -2.090     0.466     2.090
      3                (none)                          (none)    -1.983     0.179     2.090
      4                (none)                          (none)    -2.073     0.543     2.090
      5                (none)                          (none)    -2.057    -0.186     1.698
      6                (none)                          (none)    -2.090    -0.5

---

## 11. Regression with Raw SIF and Model Comparison Table

The baseline model (Section 5) uses SIF **z-scores** — pixel-month standardised anomalies. Z-scores are dimensionless and remove spatial heterogeneity in mean SIF (driven by canopy structure, crop type, and climatology), making the regression cleaner for detecting drought responses.

This section repeats the baseline regression using **raw (absolute) SIF** values in mW m⁻² sr⁻¹ nm⁻¹. Raw SIF includes both the spatial mean and the anomaly, so month fixed effects carry more of the variance and the coefficients are in physical units. Results are placed side-by-side in a comparison table for the manuscript.

**Model A (z-score):** SIF_z ~ SPEI + ΔET + SPEI×ΔET + C(month)  — *already estimated in Section 5*

**Model B (raw SIF):** SIF_raw ~ SPEI + ΔET + SPEI×ΔET + C(month)

### 11.1 Load Raw SIF from Processed Files

Raw SIF (mW m⁻² sr⁻¹ nm⁻¹) is not stored in `df_combined_gs.parquet`; it must be loaded directly from the processed OCO-2 NetCDF files and merged with `df_reg` by lat, lon, year, month.

In [16]:
import xarray as xr

_sif_proc_dir  = proc / 'sif'
_sif_raw_rows  = []
_n_loaded_sif  = 0
_n_missing_sif = 0

for year in YEARS:
    for month in GROWING_SEASON:
        yyyymm = str(year) + str(month).zfill(2)
        fa = _sif_proc_dir / ('SIF_CONUS_' + yyyymm + 'a.nc')
        fb = _sif_proc_dir / ('SIF_CONUS_' + yyyymm + 'b.nc')

        _arrs = []
        for fp in [fa, fb]:
            if not fp.exists():
                continue
            try:
                ds  = xr.open_dataset(fp)
                var = 'sif_ann' if 'sif_ann' in ds else list(ds.data_vars)[0]
                arr = ds[var].values.squeeze()
                if arr.ndim == 3:
                    arr = arr[0]
                ds.close()
                if arr.shape == (n_lat, n_lon):
                    _arrs.append(arr.astype(float))
            except Exception as e:
                print('  SIF load error:', fp.name, '-', e)

        if not _arrs:
            _n_missing_sif += 1
            continue

        _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)

        # Extract values at cropland pixel positions
        _crop_r, _crop_c = np.where(crop_mask_static)
        _sif_raw_rows.append(pd.DataFrame({
            'year':    year,
            'month':   month,
            'lat':     np.round(CONUS_LAT[_crop_r], 4),
            'lon':     np.round(CONUS_LON[_crop_c], 4),
            'sif_raw': _monthly_sif[_crop_r, _crop_c],
        }))
        _n_loaded_sif += 1

    print('  SIF loaded year', year)

df_sif_raw = pd.concat(_sif_raw_rows, ignore_index=True)
print()
print('Raw SIF extract shape:', df_sif_raw.shape)
print('Months loaded:', _n_loaded_sif, '| Missing:', _n_missing_sif)
print('SIF raw stats:')
_sv = df_sif_raw['sif_raw'].dropna()
print('  N valid: {:,}'.format(len(_sv)))
print('  mean={:.3f}  std={:.3f}  range=[{:.3f}, {:.3f}] mW/m2/sr/nm'.format(
    float(_sv.mean()), float(_sv.std()), float(_sv.min()), float(_sv.max())))

/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)


  SIF loaded year 2015
  SIF loaded year 2016


/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)


/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.

  SIF loaded year 2017
  SIF loaded year 2018


...

/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)


/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)


  SIF loaded year 2023
  SIF loaded year 2024



Raw SIF extract shape: (1484027, 5)
Months loaded: 59 | Missing: 1
SIF raw stats:
  N valid: 800,846
  mean=0.192  std=0.186  range=[-0.223, 1.056] mW/m2/sr/nm


/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)
/tmp/ipykernel_2954802/1716383280.py:34: RuntimeWarning: Mean of empty slice
  _monthly_sif = np.nanmean(np.stack(_arrs, axis=0), axis=0)


### 11.2 Merge Raw SIF and Run Model B

In [17]:
# ── Merge raw SIF onto df_reg ─────────────────────────────────────────────
_df_reg_w = df_reg.copy()
_df_reg_w['lat_r'] = _df_reg_w['lat'].round(4)
_df_reg_w['lon_r'] = _df_reg_w['lon'].round(4)

df_reg_raw = _df_reg_w.merge(
    df_sif_raw[['lat', 'lon', 'year', 'month', 'sif_raw']].rename(
        columns={'lat': 'lat_r', 'lon': 'lon_r'}
    ),
    on=['lat_r', 'lon_r', 'year', 'month'],
    how='left',
)

_raw_join_rate = df_reg_raw['sif_raw'].notna().sum()
print('Rows with raw SIF joined: {:,} / {:,} ({:.1f}%)'.format(
    _raw_join_rate, len(df_reg_raw),
    100 * _raw_join_rate / len(df_reg_raw),
))

# ── Regression-ready subset ────────────────────────────────────────────────
df_raw_reg = df_reg_raw.dropna(subset=['sif_raw', 'spei90d', 'delta_et']).copy()
df_raw_reg = df_raw_reg[
    np.isfinite(df_raw_reg['sif_raw']) &
    np.isfinite(df_raw_reg['spei90d']) &
    np.isfinite(df_raw_reg['delta_et'])
].copy()

print('Model B dataset: {:,} pixel-month observations'.format(len(df_raw_reg)))
print('SIF raw: mean={:.3f}  std={:.3f}'.format(
    float(df_raw_reg['sif_raw'].mean()),
    float(df_raw_reg['sif_raw'].std()),
))

# ── Model B: raw SIF ──────────────────────────────────────────────────────
res_raw = smf.ols(
    'sif_raw ~ spei90d + delta_et + spei_x_det + C(month)',
    data=df_raw_reg,
).fit(cov_type='HC3')

# ── Print results ─────────────────────────────────────────────────────────
_key_vars   = ['spei90d', 'delta_et', 'spei_x_det']
_row_labels = ['SPEI-90d', 'HumanET (deltaET)', 'SPEI x deltaET']

print()
print('Model B — Pooled OLS, HC3 Robust SE')
print('Response: SIF_raw (mW/m2/sr/nm)')
print('N = {:,}   R2 = {:.4f}   Adj. R2 = {:.4f}'.format(
    int(res_raw.nobs), res_raw.rsquared, res_raw.rsquared_adj,
))
print()
print('{:<26} {:>10} {:>10} {:>11}  Sig.'.format('Variable', 'Coef.', 'Std.Err.', 'p-value'))
print('-' * 66)
for v, lbl in zip(_key_vars, _row_labels):
    coef = res_raw.params[v]
    se   = res_raw.bse[v]
    pval = res_raw.pvalues[v]
    sig  = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else 'n.s.'))
    print('  {:<24} {:>10.5f} {:>10.5f} {:>11.4g}  {}'.format(lbl, coef, se, pval, sig))

print()
_b2_raw, _b3_raw = res_raw.params['delta_et'], res_raw.params['spei_x_det']
sig3_raw = '***' if res_raw.pvalues['spei_x_det'] < 0.001 else (
    '**' if res_raw.pvalues['spei_x_det'] < 0.01 else (
    '*' if res_raw.pvalues['spei_x_det'] < 0.05 else 'n.s.'))
_dir_raw = 'POSITIVE - irrigation benefit diminishes under drought' if _b3_raw > 0 else (
           'NEGATIVE - irrigation buffers SIF losses under drought')
print('b3 (interaction) = {:.5f}  ({}, p = {:.4g})'.format(
    _b3_raw, sig3_raw, float(res_raw.pvalues['spei_x_det'])))
print('Direction:', _dir_raw)

Rows with raw SIF joined: 434,245 / 434,245 (100.0%)
Model B dataset: 434,245 pixel-month observations
SIF raw: mean=0.248  std=0.195



Model B — Pooled OLS, HC3 Robust SE
Response: SIF_raw (mW/m2/sr/nm)
N = 434,245   R2 = 0.4719   Adj. R2 = 0.4719

Variable                        Coef.   Std.Err.     p-value  Sig.
------------------------------------------------------------------
  SPEI-90d                    0.02880    0.00033           0  ***
  HumanET (deltaET)          -0.00040    0.00001           0  ***
  SPEI x deltaET             -0.00040    0.00001           0  ***

b3 (interaction) = -0.00040  (***, p = 0)
Direction: NEGATIVE - irrigation buffers SIF losses under drought


### 11.3 Comparison Table (Models A and B)

Side-by-side table of baseline results. Only SPEI, ΔET, and SPEI×ΔET rows are shown (month FE omitted). Significance: *** p<0.001, ** p<0.01, * p<0.05.

In [18]:
def _sig_star(pval):
    """Return significance stars string."""
    if pval < 0.001: return '***'
    if pval < 0.01:  return '**'
    if pval < 0.05:  return '*'
    return ''

_TABLE_VARS  = ['spei90d', 'delta_et', 'spei_x_det']
_TABLE_LBLS  = ['SPEI-90d (drought)', 'HumanET (deltaET)', 'SPEI x deltaET (interaction)']
_MODELS      = [('Model A: SIF z-score', res), ('Model B: SIF raw', res_raw)]

# ── Print formatted table ─────────────────────────────────────────────────
_hdr = '{:<32}  {:>10}  {:>8}  {:>10}  {:>8}'.format(
    'Variable',
    'z-score b', 'z-score SE',
    'raw SIF b',  'raw SIF SE',
)
print(_hdr)
print('-' * 74)

for var, lbl in zip(_TABLE_VARS, _TABLE_LBLS):
    _rz  = res
    _rrw = res_raw
    _bz  = _rz.params[var]
    _sez = _rz.bse[var]
    _pz  = _rz.pvalues[var]
    _brw = _rrw.params[var]
    _srw = _rrw.bse[var]
    _prw = _rrw.pvalues[var]

    print('{:<32}  {:>10.5f}  {:>8.5f}  {:>10.5f}  {:>8.5f}  {}  {}'.format(
        lbl,
        _bz, _sez,
        _brw, _srw,
        _sig_star(_pz), _sig_star(_prw),
    ))

print('-' * 74)
print('{:<32}  {:>10}  {:>8}  {:>10}  {:>8}'.format(
    'N observations',
    '{:,}'.format(int(res.nobs)), '',
    '{:,}'.format(int(res_raw.nobs)), '',
))
print('{:<32}  {:>10.4f}  {:>8}  {:>10.4f}  {:>8}'.format(
    'R2', res.rsquared, '', res_raw.rsquared, '',
))
print('{:<32}  {:>10.4f}  {:>8}  {:>10.4f}  {:>8}'.format(
    'Adj. R2', res.rsquared_adj, '', res_raw.rsquared_adj, '',
))
print()
print('*** p<0.001   ** p<0.01   * p<0.05')
print('Month fixed effects included in both models (not shown).')
print('HC3 heteroskedasticity-robust standard errors.')

Variable                           z-score b  z-score SE   raw SIF b  raw SIF SE
--------------------------------------------------------------------------
SPEI-90d (drought)                   0.17264   0.00209     0.02880   0.00033  ***  ***
HumanET (deltaET)                    0.00357   0.00006    -0.00040   0.00001  ***  ***
SPEI x deltaET (interaction)         0.00067   0.00006    -0.00040   0.00001  ***  ***
--------------------------------------------------------------------------
N observations                       434,245               434,245          
R2                                    0.2055                0.4719          
Adj. R2                               0.2055                0.4719          

*** p<0.001   ** p<0.01   * p<0.05
Month fixed effects included in both models (not shown).
HC3 heteroskedasticity-robust standard errors.


### 11.4 Export Comparison Table — LaTeX and PNG

In [19]:
# ── Build table data ──────────────────────────────────────────────────────
def _fmt_coef(b, se, pval):
    """Format coefficient as 'b.xxxxx (SE)***' ."""
    star = _sig_star(pval)
    return '{:.5f}{}'.format(b, star), '({:.5f})'.format(se)

_rows_tex = []
for var, lbl in zip(_TABLE_VARS, _TABLE_LBLS):
    bz, sez = _fmt_coef(res.params[var], res.bse[var], res.pvalues[var])
    br, ser = _fmt_coef(res_raw.params[var], res_raw.bse[var], res_raw.pvalues[var])
    _rows_tex.append((lbl, bz, sez, br, ser))

# ── LaTeX table ───────────────────────────────────────────────────────────
_latex_lines = [
    r'\begin{table}[htbp]',
    r'\centering',
    r'\caption{Regression Results: SIF Response to Drought and Irrigation (CONUS Cropland, 2015--2024)}',
    r'\label{tab:sif_regression}',
    r'\begin{tabular}{lcccc}',
    r'\hline',
    r' & \multicolumn{2}{c}{Model A: SIF z-score} & \multicolumn{2}{c}{Model B: SIF raw (mW\,m$^{-2}$\,sr$^{-1}$\,nm$^{-1}$)} \\',
    r' & $\hat{\beta}$ & SE & $\hat{\beta}$ & SE \\',
    r'\hline',
]

for lbl, bz, sez, br, ser in _rows_tex:
    _latex_lines.append(
        r'{} & {} & {} & {} & {} \\'.format(lbl, bz, sez, br, ser)
    )

_latex_lines += [
    r'\hline',
    r'$N$ & \multicolumn{{2}}{{c}}{{{:,}}} & \multicolumn{{2}}{{c}}{{{:,}}} \\'.format(
        int(res.nobs), int(res_raw.nobs)
    ),
    r'$R^2$ & \multicolumn{{2}}{{c}}{{{:.4f}}} & \multicolumn{{2}}{{c}}{{{:.4f}}} \\'.format(
        res.rsquared, res_raw.rsquared
    ),
    r'Adj.\ $R^2$ & \multicolumn{{2}}{{c}}{{{:.4f}}} & \multicolumn{{2}}{{c}}{{{:.4f}}} \\'.format(
        res.rsquared_adj, res_raw.rsquared_adj
    ),
    r'\hline',
    r'\multicolumn{5}{l}{\footnotesize Note: Pooled OLS with month fixed effects (not shown). HC3 robust standard errors.} \\',
    r'\multicolumn{5}{l}{\footnotesize $^{***}$p$<$0.001; $^{**}$p$<$0.01; $^{*}$p$<$0.05.} \\',
    r'\end{tabular}',
    r'\end{table}',
]

_latex_str = '\n'.join(_latex_lines)
_tex_path = figs / 'table_regression_comparison.tex'
with open(_tex_path, 'w') as f:
    f.write(_latex_str)
print('LaTeX table saved:', _tex_path)
print()
print(_latex_str)

LaTeX table saved: /home/pielab-sandbox-jcoldiron/SIF-Analysis/figures/conus/table_regression_comparison.tex

\begin{table}[htbp]
\centering
\caption{Regression Results: SIF Response to Drought and Irrigation (CONUS Cropland, 2015--2024)}
\label{tab:sif_regression}
\begin{tabular}{lcccc}
\hline
 & \multicolumn{2}{c}{Model A: SIF z-score} & \multicolumn{2}{c}{Model B: SIF raw (mW\,m$^{-2}$\,sr$^{-1}$\,nm$^{-1}$)} \\
 & $\hat{\beta}$ & SE & $\hat{\beta}$ & SE \\
\hline
SPEI-90d (drought) & 0.17264*** & (0.00209) & 0.02880*** & (0.00033) \\
HumanET (deltaET) & 0.00357*** & (0.00006) & -0.00040*** & (0.00001) \\
SPEI x deltaET (interaction) & 0.00067*** & (0.00006) & -0.00040*** & (0.00001) \\
\hline
$N$ & \multicolumn{2}{c}{434,245} & \multicolumn{2}{c}{434,245} \\
$R^2$ & \multicolumn{2}{c}{0.2055} & \multicolumn{2}{c}{0.4719} \\
Adj.\ $R^2$ & \multicolumn{2}{c}{0.2055} & \multicolumn{2}{c}{0.4719} \\
\hline
\multicolumn{5}{l}{\footnotesize Note: Pooled OLS with month fixed effects (not 

In [20]:
# ── PNG table figure ──────────────────────────────────────────────────────
_col_headers = [
    'Variable',
    'z-score\u03b2', 'z-score SE',
    'raw SIF \u03b2', 'raw SIF SE',
]

_tbl_data = []
for var, lbl in zip(_TABLE_VARS, _TABLE_LBLS):
    bz, sez = _fmt_coef(res.params[var], res.bse[var], res.pvalues[var])
    br, ser = _fmt_coef(res_raw.params[var], res_raw.bse[var], res_raw.pvalues[var])
    _tbl_data.append([lbl, bz, sez, br, ser])

# Footer rows (no SE columns for fit statistics)
_tbl_data.append(['', '', '', '', ''])
_tbl_data.append(['N observations',
                  '{:,}'.format(int(res.nobs)), '',
                  '{:,}'.format(int(res_raw.nobs)), ''])
_tbl_data.append(['R\u00b2',
                  '{:.4f}'.format(res.rsquared), '',
                  '{:.4f}'.format(res_raw.rsquared), ''])
_tbl_data.append(['Adj. R\u00b2',
                  '{:.4f}'.format(res.rsquared_adj), '',
                  '{:.4f}'.format(res_raw.rsquared_adj), ''])

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.axis('off')

tbl = ax.table(
    cellText=_tbl_data,
    colLabels=_col_headers,
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1],
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)

# Style: bold header row
for j in range(len(_col_headers)):
    tbl[(0, j)].set_facecolor('#2C3E50')
    tbl[(0, j)].set_text_props(color='white', fontweight='bold')

# Alternate row shading
for i in range(1, len(_tbl_data) + 1):
    for j in range(len(_col_headers)):
        if i in (len(_tbl_data) - 2, len(_tbl_data) - 1, len(_tbl_data)):
            tbl[(i, j)].set_facecolor('#ECF0F1')
        elif i % 2 == 0:
            tbl[(i, j)].set_facecolor('#F8F9FA')
        else:
            tbl[(i, j)].set_facecolor('white')
        tbl[(i, j)].set_text_props(ha='center')

# Left-align the variable name column
for i in range(1, len(_tbl_data) + 1):
    tbl[(i, 0)].set_text_props(ha='left')

ax.text(0.0, -0.04,
        '*** p<0.001  ** p<0.01  * p<0.05  |  Month FE included (not shown)  |  HC3 robust SE',
        transform=ax.transAxes, fontsize=7, color='#555555', va='top')

fig.suptitle(
    'Table 1: SIF Regression Comparison  (Models A and B, CONUS Cropland 2015-2024)',
    fontsize=10, fontweight='bold', y=1.02,
)

plt.tight_layout()
_tbl_stem = figs / 'table_regression_comparison'
plt.savefig(str(_tbl_stem) + '.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.savefig(str(_tbl_stem) + '.pdf', bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print('Saved: table_regression_comparison.png / .pdf  (300 DPI)')

Saved: table_regression_comparison.png / .pdf  (300 DPI)


---

## 12. Models C and D — NLDAS FORA Atmospheric Forcing Variables

**Prerequisite:** Run `src/scripts/download/20_download_nldas_fora.py` first to download and process the NLDAS FORA monthly files into `data/processed/conus/nldas_fora/FORA_YYYYMM.nc`.

This section adds three FORA-derived atmospheric predictors:

| Variable | Description | Computed from |
|---|---|---|
| `Tair` | 2-m air temperature (K) | TMP directly |
| `VPD` | Vapour pressure deficit (kPa) | TMP + SPFH + PRES |
| `Wind` | Wind speed (m/s) | sqrt(UGRD² + VGRD²) |

**VPD formula (Tetens equation):**
- T_C = Tair − 273.15  
- es = 0.6108 × exp(17.27 × T_C / (T_C + 237.3))  [kPa]  
- ea = Qair × PSurf / (0.622 + 0.378 × Qair)       [kPa]  
- VPD = max(es − ea, 0)                             [kPa]

**Models:**

**Model C (z-score + FORA met):**  
SIF_z ~ SPEI + ΔET + SPEI×ΔET + Tair + VPD + Wind + C(month)

**Model D (raw SIF + FORA met):**  
SIF_raw ~ SPEI + ΔET + SPEI×ΔET + Tair + VPD + Wind + C(month)

Note: before fitting, a predictor correlation heatmap checks for collinearity among SPEI, ΔET, Tair, VPD, and Wind. Pairs with |r| > 0.70 are flagged.

### 12.1 Load FORA Variables

In [21]:
_fora_dir = proc / 'nldas_fora'

if not _fora_dir.exists() or not list(_fora_dir.glob('FORA_*.nc')):
    print('FORA data not found at:', _fora_dir)
    print('Run src/scripts/download/20_download_nldas_fora.py first.')
    print('Skipping Models C and D.')
    _fora_available = False
else:
    _fora_available = True
    _n_lat, _n_lon = len(CONUS_LAT), len(CONUS_LON)

    if crop_mask_static is not None:
        _crop_r_f, _crop_c_f = np.where(crop_mask_static)
    else:
        raise RuntimeError('crop_mask_static required for FORA extraction')

    _fora_rows = []
    for year in YEARS:
        for month in GROWING_SEASON:
            yyyymm = str(year) + str(month).zfill(2)
            fp = _fora_dir / ('FORA_' + yyyymm + '.nc')
            if not fp.exists():
                print('  Missing FORA file:', fp.name)
                continue
            try:
                ds = xr.open_dataset(fp)
                _row = {
                    'year': year, 'month': month,
                    'lat':  np.round(CONUS_LAT[_crop_r_f], 4),
                    'lon':  np.round(CONUS_LON[_crop_c_f], 4),
                }
                for vname in ['TMP', 'SPFH', 'PRES', 'UGRD', 'VGRD', 'DSWRF']:
                    if vname in ds:
                        arr = ds[vname].values.squeeze()
                        if arr.shape == (_n_lat, _n_lon):
                            _row[vname] = arr[_crop_r_f, _crop_c_f]
                        else:
                            _row[vname] = np.full(len(_crop_r_f), np.nan)
                    else:
                        _row[vname] = np.full(len(_crop_r_f), np.nan)
                ds.close()
                _fora_rows.append(pd.DataFrame(_row))
            except Exception as e:
                print('  FORA load error', fp.name, ':', e)
        print('  FORA loaded year', year)

    df_fora = pd.concat(_fora_rows, ignore_index=True)
    print()
    print('FORA extract shape:', df_fora.shape)
    for v in ['TMP', 'SPFH', 'PRES', 'UGRD', 'VGRD', 'DSWRF']:
        if v in df_fora:
            arr = df_fora[v].dropna()
            print('  {:6s}  mean={:.3g}  min={:.3g}  max={:.3g}'.format(
                v, float(arr.mean()), float(arr.min()), float(arr.max())))

FORA data not found at: /home/pielab-sandbox-jcoldiron/SIF-Analysis/data/processed/conus/nldas_fora
Run src/scripts/download/20_download_nldas_fora.py first.
Skipping Models C and D.


### 12.2 Compute Derived Variables (VPD, Wind Speed) and Merge

In [22]:
if _fora_available:
    # ── Tair (K) ──────────────────────────────────────────────────────────
    df_fora['Tair'] = df_fora['TMP'].copy()

    # ── Wind speed (m/s) ─────────────────────────────────────────────────
    df_fora['Wind'] = np.sqrt(df_fora['UGRD']**2 + df_fora['VGRD']**2)

    # ── VPD (kPa) — Tetens equation ──────────────────────────────────────
    # Saturation vapour pressure [kPa]
    _T_C = df_fora['TMP'] - 273.15
    _es  = 0.6108 * np.exp(17.27 * _T_C / (_T_C + 237.3))
    # Actual vapour pressure [kPa]
    _Qair  = df_fora['SPFH']
    _PSurf = df_fora['PRES']
    _ea  = _Qair * _PSurf / (1000.0 * (0.622 + 0.378 * _Qair))  # PRES in Pa → /1000 for kPa
    df_fora['VPD'] = np.maximum(_es - _ea, 0.0)

    print('Derived variables:')
    for v in ['Tair', 'Wind', 'VPD']:
        arr = df_fora[v].dropna()
        print('  {:6s}  mean={:.3g}  min={:.3g}  max={:.3g}'.format(
            v, float(arr.mean()), float(arr.min()), float(arr.max())))
    print()

    # ── Merge FORA variables onto df_reg (z-score model) and df_raw_reg ──
    _df_fora_merge = df_fora[['lat', 'lon', 'year', 'month',
                               'Tair', 'VPD', 'Wind']].copy()
    _df_fora_merge['lat_r'] = _df_fora_merge['lat'].round(4)
    _df_fora_merge['lon_r'] = _df_fora_merge['lon'].round(4)
    _df_fora_merge = _df_fora_merge.drop(columns=['lat', 'lon'])

    # Merge onto existing df_expanded (from Section 9, which already has NLDAS Noah vars)
    try:
        _base_for_c = df_expanded.copy()
    except NameError:
        # df_expanded may not exist if Section 9 was skipped
        _base_for_c = df_reg.copy()

    _base_for_c['lat_r'] = _base_for_c['lat'].round(4)
    _base_for_c['lon_r'] = _base_for_c['lon'].round(4)

    df_fora_model = _base_for_c.merge(
        _df_fora_merge, on=['lat_r', 'lon_r', 'year', 'month'], how='left'
    )

    # Merge raw SIF for Model D
    _df_raw_sif = df_reg_raw[['lat_r', 'lon_r', 'year', 'month', 'sif_raw']].copy() \
        if 'df_reg_raw' in dir() and 'sif_raw' in df_reg_raw.columns \
        else pd.DataFrame(columns=['lat_r', 'lon_r', 'year', 'month', 'sif_raw'])

    df_fora_model_raw = df_fora_model.merge(
        _df_raw_sif, on=['lat_r', 'lon_r', 'year', 'month'], how='left'
    )

    _fora_joined = df_fora_model['Tair'].notna().sum()
    print('FORA merge: {:,} / {:,} rows have Tair ({:.1f}%)'.format(
        _fora_joined, len(df_fora_model),
        100 * _fora_joined / len(df_fora_model),
    ))
else:
    print('Skipping — FORA data not available.')

Skipping — FORA data not available.


### 12.3 Predictor Correlation Matrix (FORA + SPEI + ΔET)

Flag any predictor pair with |r| > 0.70 as potentially collinear before fitting Models C and D.

In [23]:
if _fora_available:
    _FORA_CORR_COLS = ['spei90d', 'delta_et', 'Tair', 'VPD', 'Wind']
    _df_corr = df_fora_model[_FORA_CORR_COLS].dropna()
    print('Complete cases for FORA correlation matrix: {:,}'.format(len(_df_corr)))

    _corr_fora = _df_corr.corr(method='pearson')
    print()
    print('Pearson correlation matrix:')
    print(_corr_fora.round(3).to_string())
    print()

    _HIGH_R = 0.70
    print('Pairs with |r| > {}:'.format(_HIGH_R))
    _found_high = False
    for i, c1 in enumerate(_FORA_CORR_COLS):
        for j, c2 in enumerate(_FORA_CORR_COLS):
            if j <= i:
                continue
            r = _corr_fora.loc[c1, c2]
            if abs(r) > _HIGH_R:
                print('  {} - {}:  r = {:.3f}  *** potential collinearity'.format(c1, c2, r))
                _found_high = True
    if not _found_high:
        print('  (none above {} threshold)'.format(_HIGH_R))

    # ── Heatmap ──────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(6, 5))
    _cmat = _corr_fora.values
    _n    = len(_FORA_CORR_COLS)

    im = ax.imshow(_cmat, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    cb = plt.colorbar(im, ax=ax, shrink=0.85)
    cb.set_label('Pearson r', fontsize=9)

    ax.set_xticks(range(_n))
    ax.set_yticks(range(_n))
    ax.set_xticklabels(_FORA_CORR_COLS, rotation=30, ha='right', fontsize=9)
    ax.set_yticklabels(_FORA_CORR_COLS, fontsize=9)

    for i in range(_n):
        for j in range(_n):
            _r = _cmat[i, j]
            ax.text(j, i, '{:.2f}'.format(_r), ha='center', va='center', fontsize=8,
                    color='white' if abs(_r) > 0.7 else 'black')
            if i != j and abs(_r) > _HIGH_R:
                ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1,
                             fill=False, edgecolor='black', lw=2.0))

    ax.set_title('FORA + Core Predictor Correlation Matrix (|r|>0.70 boxed)', fontsize=10)
    plt.tight_layout()
    plt.savefig(figs / 'supp_correlation_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: supp_correlation_matrix.png')
else:
    print('Skipping — FORA data not available.')

Skipping — FORA data not available.


### 12.4 Fit Models C and D

**Model C:** SIF_z ~ SPEI + ΔET + SPEI×ΔET + Tair + VPD + Wind + C(month)  
**Model D:** SIF_raw ~ SPEI + ΔET + SPEI×ΔET + Tair + VPD + Wind + C(month)

> **Note on collinearity:** If Tair and VPD show |r| > 0.70 in the matrix above, consider running model variants that exclude one of them. Coefficient interpretation is valid even under moderate collinearity, but standard errors will be inflated for the correlated pair.

In [24]:
if _fora_available:
    _FORA_REG_COLS_C = ['sif_z', 'spei90d', 'delta_et', 'spei_x_det', 'month',
                        'Tair', 'VPD', 'Wind']
    _FORA_REG_COLS_D = ['sif_raw', 'spei90d', 'delta_et', 'spei_x_det', 'month',
                        'Tair', 'VPD', 'Wind']

    df_c = df_fora_model[_FORA_REG_COLS_C].dropna().copy()
    df_c = df_c[np.isfinite(df_c['sif_z'])].copy()
    print('Model C dataset: {:,} obs'.format(len(df_c)))

    df_d = df_fora_model_raw[_FORA_REG_COLS_D].dropna().copy()
    df_d = df_d[np.isfinite(df_d['sif_raw'])].copy()
    print('Model D dataset: {:,} obs'.format(len(df_d)))

    _formula_fora = ('sif_z ~ spei90d + delta_et + spei_x_det'
                     ' + Tair + VPD + Wind + C(month)')
    _formula_fora_raw = ('sif_raw ~ spei90d + delta_et + spei_x_det'
                         ' + Tair + VPD + Wind + C(month)')

    res_c = smf.ols(_formula_fora,     data=df_c).fit(cov_type='HC3')
    res_d = smf.ols(_formula_fora_raw, data=df_d).fit(cov_type='HC3')

    print()
    _key_vars_cd  = ['spei90d', 'delta_et', 'spei_x_det', 'Tair', 'VPD', 'Wind']
    _key_lbls_cd  = ['SPEI-90d', 'HumanET (deltaET)', 'SPEI x deltaET',
                     'Tair (K)', 'VPD (kPa)', 'Wind (m/s)']

    for _model_lbl, _res_m, _resp in [
        ('Model C — SIF z-score + FORA', res_c, 'sif_z'),
        ('Model D — SIF raw + FORA',     res_d, 'sif_raw'),
    ]:
        print('=' * 60)
        print(_model_lbl)
        print('N = {:,}   R2 = {:.4f}   Adj. R2 = {:.4f}'.format(
            int(_res_m.nobs), _res_m.rsquared, _res_m.rsquared_adj))
        print()
        print('{:<26} {:>10} {:>10} {:>11}  Sig.'.format(
            'Variable', 'Coef.', 'Std.Err.', 'p-value'))
        print('-' * 66)
        for v, lbl in zip(_key_vars_cd, _key_lbls_cd):
            if v not in _res_m.params:
                continue
            coef = _res_m.params[v]
            se   = _res_m.bse[v]
            pval = _res_m.pvalues[v]
            sig  = _sig_star(pval)
            print('  {:<24} {:>10.5f} {:>10.5f} {:>11.4g}  {}'.format(
                lbl, coef, se, pval, sig))
        print()
else:
    print('Skipping — FORA data not available.')

Skipping — FORA data not available.


### 12.5 Extended Four-Model Comparison Table

Extends the two-model table from Section 11.3 with Models C and D. LaTeX and PNG versions saved to `figures/conus/`.

In [25]:
if _fora_available:
    # ── Build 4-model table data ──────────────────────────────────────────
    _MODELS_4 = [
        ('A: SIF z',      res),
        ('B: SIF raw',    res_raw),
        ('C: SIF z+FORA', res_c),
        ('D: SIF raw+FORA', res_d),
    ]
    _TABLE_VARS_4  = ['spei90d', 'delta_et', 'spei_x_det', 'Tair', 'VPD', 'Wind']
    _TABLE_LBLS_4  = ['SPEI-90d', 'HumanET (deltaET)', 'SPEI x deltaET',
                      'Tair (K)', 'VPD (kPa)', 'Wind (m/s)']

    # Print text version
    _hdr4 = '{:<28}'.format('Variable')
    for lbl, _ in _MODELS_4:
        _hdr4 += '  {:>10}  {:>8}'.format(lbl + ' b', lbl + ' SE')
    print(_hdr4)
    print('-' * (28 + 4 * 22))

    for var, lbl in zip(_TABLE_VARS_4, _TABLE_LBLS_4):
        _line = '{:<28}'.format(lbl)
        for _, _res_m in _MODELS_4:
            if var in _res_m.params:
                _bv, _sev = _fmt_coef(_res_m.params[var], _res_m.bse[var], _res_m.pvalues[var])
            else:
                _bv, _sev = '—', '—'
            _line += '  {:>10}  {:>8}'.format(_bv, _sev)
        print(_line)

    print('-' * (28 + 4 * 22))
    _nline = '{:<28}'.format('N observations')
    _r2line = '{:<28}'.format('R2')
    _adj_r2line = '{:<28}'.format('Adj. R2')
    for _, _res_m in _MODELS_4:
        _nline     += '  {:>20}'.format('{:,}'.format(int(_res_m.nobs)))
        _r2line    += '  {:>20}'.format('{:.4f}'.format(_res_m.rsquared))
        _adj_r2line += '  {:>20}'.format('{:.4f}'.format(_res_m.rsquared_adj))
    print(_nline)
    print(_r2line)
    print(_adj_r2line)

    # ── LaTeX 4-model table ───────────────────────────────────────────────
    _ltx4 = [
        r'\begin{table}[htbp]',
        r'\centering',
        r'\footnotesize',
        r'\caption{SIF Regression: Four-Model Comparison (CONUS Cropland, 2015--2024)}',
        r'\label{tab:sif_4model}',
        r'\begin{tabular}{l' + 'cc' * 4 + '}',
        r'\hline',
        r' & \multicolumn{2}{c}{A: SIF z-score} & \multicolumn{2}{c}{B: SIF raw}'
        r' & \multicolumn{2}{c}{C: SIF z+FORA} & \multicolumn{2}{c}{D: SIF raw+FORA} \\',
        r' & $\hat{\beta}$ & SE & $\hat{\beta}$ & SE & $\hat{\beta}$ & SE & $\hat{\beta}$ & SE \\',
        r'\hline',
    ]
    for var, lbl in zip(_TABLE_VARS_4, _TABLE_LBLS_4):
        _row_cells = [lbl]
        for _, _res_m in _MODELS_4:
            if var in _res_m.params:
                _bv, _sev = _fmt_coef(_res_m.params[var], _res_m.bse[var], _res_m.pvalues[var])
            else:
                _bv, _sev = r'\multicolumn{2}{c}{---}', ''
            _row_cells += [_bv, _sev]
        _ltx4.append(' & '.join(_row_cells) + r' \\')

    _ltx4 += [
        r'\hline',
        '$N$ & ' + ' & '.join(
            [r'\multicolumn{{2}}{{c}}{{{:,}}}'.format(int(_r.nobs)) for _, _r in _MODELS_4]
        ) + r' \\',
        '$R^2$ & ' + ' & '.join(
            [r'\multicolumn{{2}}{{c}}{{{:.4f}}}'.format(_r.rsquared) for _, _r in _MODELS_4]
        ) + r' \\',
        r'\hline',
        r'\multicolumn{9}{l}{\footnotesize HC3 robust SE; month FE in all models.} \\',
        r'\multicolumn{9}{l}{\footnotesize $^{***}$p$<$0.001; $^{**}$p$<$0.01; $^{*}$p$<$0.05.} \\',
        r'\end{tabular}',
        r'\end{table}',
    ]

    _ltx4_str = '\n'.join(_ltx4)
    _tex4_path = figs / 'table_regression_comparison.tex'
    with open(_tex4_path, 'w') as f:
        f.write(_ltx4_str)

    # ── PNG table figure ──────────────────────────────────────────────────
    _col_hdrs4 = ['Variable'] + [
        m + ' b' for m, _ in _MODELS_4
    ] + [m + ' SE' for m, _ in _MODELS_4]

    # Interleave columns: A-b, A-SE, B-b, B-SE, C-b, C-SE, D-b, D-SE
    _col_hdrs4 = ['Variable', 'A b', 'A SE', 'B b', 'B SE', 'C b', 'C SE', 'D b', 'D SE']
    _tbl4 = []
    for var, lbl in zip(_TABLE_VARS_4, _TABLE_LBLS_4):
        _r = [lbl]
        for _, _res_m in _MODELS_4:
            if var in _res_m.params:
                _bv, _sev = _fmt_coef(_res_m.params[var], _res_m.bse[var], _res_m.pvalues[var])
            else:
                _bv, _sev = '—', '—'
            _r += [_bv, _sev]
        _tbl4.append(_r)

    _tbl4.append([''] * 9)
    _tbl4.append(['N'] + [item for _, _res_m in _MODELS_4
                          for item in ['{:,}'.format(int(_res_m.nobs)), '']])
    _tbl4.append(['R\u00b2'] + [item for _, _res_m in _MODELS_4
                                for item in ['{:.4f}'.format(_res_m.rsquared), '']])

    fig, ax = plt.subplots(figsize=(13, 3.5))
    ax.axis('off')
    tbl4 = ax.table(cellText=_tbl4, colLabels=_col_hdrs4,
                    cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
    tbl4.auto_set_font_size(False)
    tbl4.set_fontsize(7.5)
    for j in range(len(_col_hdrs4)):
        tbl4[(0, j)].set_facecolor('#2C3E50')
        tbl4[(0, j)].set_text_props(color='white', fontweight='bold')
    for i in range(1, len(_tbl4) + 1):
        for j in range(len(_col_hdrs4)):
            tbl4[(i, j)].set_facecolor('#F8F9FA' if i % 2 == 0 else 'white')
            tbl4[(i, j)].set_text_props(ha='center')
        tbl4[(i, 0)].set_text_props(ha='left')

    ax.text(0.0, -0.04,
            '*** p<0.001  ** p<0.01  * p<0.05  |  HC3 robust SE  |  Month FE in all models  '
            '|  FORA models require 20_download_nldas_fora.py',
            transform=ax.transAxes, fontsize=6.5, color='#555555', va='top')
    fig.suptitle('Table 2: Four-Model Comparison  (CONUS Cropland 2015-2024)',
                 fontsize=9, fontweight='bold', y=1.02)
    plt.tight_layout()
    _tbl4_stem = figs / 'table_regression_comparison'
    plt.savefig(str(_tbl4_stem) + '.png', dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.savefig(str(_tbl4_stem) + '.pdf', bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.show()
    print('Saved: table_regression_comparison.tex / .png / .pdf')
else:
    print('Models C and D not available — run 20_download_nldas_fora.py first.')
    print('Once FORA data is downloaded, re-run Section 12 to populate this table.')

Models C and D not available — run 20_download_nldas_fora.py first.
Once FORA data is downloaded, re-run Section 12 to populate this table.
